# 🛰️ AFETSONAR — Notebook 2_v2: Öğretmen Modeli Eğitimi v2

**Calamitas AI · Teknofest 2025 · Tam Refactor**

---

## 📌 Bu Notebook Ne Yapıyor?

Bu, öğretmen modelinin **tamamen yenilenmiş** eğitim notebook'u. İlk eğitimimiz (`mIoU_no_bg = 0.298`) bize ne öğretti, o bilgilerle 8 kritik iyileştirme uyguluyoruz.

### 🔧 Uygulanan 8 İyileştirme

| # | Değişiklik | Neden? |
|---|---|---|
| 1 | **6 sınıf** (un-classified=5 eklendi) | Notebook 1b'de düzeltildi, veri artık temiz |
| 2 | **True Siamese mimari** | Eski model pre kanallarını yoksayıyordu. Şimdi pre ve post için paylaşılan encoder, sonra concat+diff fusion |
| 3 | **SegFormer-B3** (B5 yerine) | Daha hızlı eğitim, daha az overfitting riski, edge-ready |
| 4 | **Combo Loss** (Focal + Dice + Boundary) | Tek başına Focal yetmedi. Dice boundary quality, Boundary Loss bina kenarları için |
| 5 | **Agresif class weights** | Veri analizi: minor %0.53, destroyed %0.34. Weights: [0.05, 1, 8.5, 6.5, 10, 0.5] |
| 6 | **WeightedRandomSampler** | Hasarlı görüntüleri ~5-15x daha sık örnekle |
| 7 | **Building-aware crop** | Crop'un %80'i bina merkezli (random yerine) |
| 8 | **Sınıf-koruyucu augmentation** | GaussNoise ve CoarseDropout kapatıldı (hasar sinyalini siliyor), sadece geometric + hafif color jitter |

### 📊 Beklenen Sonuçlar

- Eğitim süresi: **~2-3 saat** (H100'de, B3 B5'ten 2x hızlı)
- Hedef mIoU_no_bg: **≥ 0.55** (ilk iyileştirme adımı)
- Eğer 0.55 aşılırsa → Notebook 2_v3 (Phase 2) ile daha da iyileştirme
- Hedef 0.78 için belki Phase 2 gerekebilir

### ⚠️ Önemli

Bu notebook **Notebook 1b çıktılarına bağımlıdır**:
- `data/splits/train_v2.csv` (sample_weight column'lu)
- `data/splits/val_v2.csv`
- Yeni maskeler (`data/processed/masks/` 6 sınıflı)

Eğer Notebook 1b'yi çalıştırmadıysan **önce onu çalıştır**, sonra buna gel.

---

## 1️⃣ Drive + Yollar + GPU

Her şey Notebook 1b'deki yapıyı varsayıyor.

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

PROJECT_ROOT = "/content/drive/MyDrive/AFETSONAR"
DATA_RAW     = os.path.join(PROJECT_ROOT, "data/raw/xview2")
DATA_MASKS   = os.path.join(PROJECT_ROOT, "data/processed/masks")
DATA_SPLITS  = os.path.join(PROJECT_ROOT, "data/splits")
SRC_DIR      = os.path.join(PROJECT_ROOT, "src")
CKPT_TEACHER = os.path.join(PROJECT_ROOT, "checkpoints/teacher")
OUTPUTS_VIZ  = os.path.join(PROJECT_ROOT, "outputs/visualizations")

# Sanity check: v2 splits var mı?
required = {
    "train_v2.csv": os.path.join(DATA_SPLITS, "train_v2.csv"),
    "val_v2.csv":   os.path.join(DATA_SPLITS, "val_v2.csv"),
    "test_v2.csv":  os.path.join(DATA_SPLITS, "test_v2.csv"),
}

print("✅ Drive bağlandı")
print()
print("📁 Sanity check (Notebook 1b çıktıları):")
all_ok = True
for name, path in required.items():
    exists = os.path.exists(path)
    marker = "✅" if exists else "❌"
    print(f"  {marker} {name}")
    if not exists:
        all_ok = False

if not all_ok:
    raise FileNotFoundError(
        "❌ Notebook 1b çıktıları eksik! Önce 01b_regenerate_masks.ipynb'yi çalıştır."
    )

os.makedirs(CKPT_TEACHER, exist_ok=True)
os.makedirs(OUTPUTS_VIZ, exist_ok=True)

# === GPU ===
import torch

print("\n⚙️  Donanım:")
if not torch.cuda.is_available():
    raise RuntimeError("GPU yok!")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1)
bf16_supported = torch.cuda.is_bf16_supported()

print(f"  GPU: {gpu_name}")
print(f"  VRAM: {vram_gb} GB")
print(f"  bf16: {'✅' if bf16_supported else '❌'}")

# H100 / A100 / L4 / T4 için gerçekçi ayarlar
# NOT: Siamese mimari encoder'dan 2 kez geçiyor → ~2x bellek
# Bu yüzden batch size'ları v1'den AZALTILDI
# Efektif batch size = BATCH_SIZE × GRAD_ACCUM_STEPS
if "H100" in gpu_name:
    BATCH_SIZE = 16
    GRAD_ACCUM_STEPS = 3   # efektif batch = 48
    NUM_WORKERS = 8
    USE_GRADIENT_CHECKPOINTING = False  # H100'de gerek yok, batch 16 zaten rahat
elif "A100" in gpu_name:
    BATCH_SIZE = 8
    GRAD_ACCUM_STEPS = 4   # efektif batch = 32
    NUM_WORKERS = 6
    USE_GRADIENT_CHECKPOINTING = True
elif "L4" in gpu_name or "V100" in gpu_name:
    BATCH_SIZE = 4
    GRAD_ACCUM_STEPS = 4
    NUM_WORKERS = 4
    USE_GRADIENT_CHECKPOINTING = True
elif "T4" in gpu_name:
    BATCH_SIZE = 2
    GRAD_ACCUM_STEPS = 8
    NUM_WORKERS = 2
    USE_GRADIENT_CHECKPOINTING = True
else:
    BATCH_SIZE = 4
    GRAD_ACCUM_STEPS = 4
    NUM_WORKERS = 4
    USE_GRADIENT_CHECKPOINTING = True

EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRAD_ACCUM_STEPS
PRECISION = "bf16" if bf16_supported else "fp16"
device = torch.device("cuda")

print(f"\n📊 Ayarlar (Siamese mimari için optimize):")
print(f"  micro batch        = {BATCH_SIZE}")
print(f"  grad accum steps   = {GRAD_ACCUM_STEPS}")
print(f"  effective batch    = {EFFECTIVE_BATCH_SIZE}")
print(f"  num_workers        = {NUM_WORKERS}")
print(f"  precision          = {PRECISION}")
print(f"  grad checkpointing = {USE_GRADIENT_CHECKPOINTING}")

Mounted at /content/drive
✅ Drive bağlandı

📁 Sanity check (Notebook 1b çıktıları):
  ✅ train_v2.csv
  ✅ val_v2.csv
  ✅ test_v2.csv

⚙️  Donanım:
  GPU: NVIDIA H100 80GB HBM3
  VRAM: 79.2 GB
  bf16: ✅

📊 Ayarlar (Siamese mimari için optimize):
  micro batch        = 16
  grad accum steps   = 3
  effective batch    = 48
  num_workers        = 8
  precision          = bf16
  grad checkpointing = False


## 2️⃣ v2 Modüllerini `src/`'a Yaz

Eski modüller (`dataset.py`, `models.py`, vs.) Drive'da hala var. Biz onların üzerine yazmayacağız — yenileri `_v2` son ekiyle ayrı dosyalar olacak:

- `src/models_v2.py` — SiameseTeacherSegformer (B3)
- `src/losses_v2.py` — ComboDamageLoss + TeacherLossV2
- `src/dataset_v2.py` — XBDDatasetV2 (building-aware crop)
- `src/augmentations_v2.py` — Hafif, sınıf-koruyan augmentation
- `src/metrics.py` — Yeniden yazılmadı, aynı kalıyor

Bu sayede eski Notebook 2 hala çalışabilir, yeni v2 ile karşılaştırma yapabilirsin.

In [2]:
import os
import base64

MODELS_V2 = base64.b64decode("IiIiCkFGRVRTT05BUiBUZWFjaGVyIE1vZGVsIHYyIOKAlCBUcnVlIFNpYW1lc2UgU2VnRm9ybWVyLUIzLgoKS2V5IGRpZmZlcmVuY2VzIGZyb20gdjE6Ci0gVHdvIHNlcGFyYXRlIGVuY29kZXJzIGZvciBwcmUgYW5kIHBvc3QgKHRydWUgU2lhbWVzZSkKLSBGZWF0dXJlIGZ1c2lvbiB2aWEgY29uY2F0ZW5hdGlvbiArIHN1YnRyYWN0aW9uIGF0IGVhY2ggc3RhZ2UKLSBTZWdGb3JtZXItQjMgYmFja2JvbmUgKDQ1TSBwYXJhbXMsIGVkZ2UtY2FwYWJsZSkKLSA2IGRhbWFnZSBjbGFzc2VzIChhZGRzIHVuLWNsYXNzaWZpZWQ9NSkKLSAzIHRhc2sgaGVhZHM6IGRhbWFnZSwgY2hhbmdlLCBkaXNhc3RlcgoiIiIKCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgpmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgU2VnZm9ybWVyRm9yU2VtYW50aWNTZWdtZW50YXRpb24sIFNlZ2Zvcm1lckNvbmZpZwoKCmNsYXNzIFNpYW1lc2VUZWFjaGVyU2VnZm9ybWVyKG5uLk1vZHVsZSk6CiAgICAiIiIKICAgIFRydWUgU2lhbWVzZSBTZWdGb3JtZXItQjMgd2l0aCBkdWFsIGVuY29kZXIgYnJhbmNoZXMuCgogICAgQXJjaGl0ZWN0dXJlOgogICAgICAgIHByZSAgW0IsIDMsIEgsIFddIC0+IGVuY29kZXJfc2hhcmVkIC0+IGZlYXR1cmVzX3ByZSAgKDQgc3RhZ2VzKQogICAgICAgIHBvc3QgW0IsIDMsIEgsIFddIC0+IGVuY29kZXJfc2hhcmVkIC0+IGZlYXR1cmVzX3Bvc3QgKDQgc3RhZ2VzKQoKICAgICAgICBmdXNpb25baV0gPSBjb25jYXQoZmVhdHVyZXNfcHJlW2ldLCBmZWF0dXJlc19wb3N0W2ldLCB8cG9zdCAtIHByZXxbaV0pCiAgICAgICAgICAgICAgICAgIC0+IDF4MSBjb252IC0+IGZ1c2VkW2ldCgogICAgICAgIGZ1c2VkIGZlYXR1cmVzIC0+IGRlY29kZV9oZWFkIC0+IGRhbWFnZV9sb2dpdHMKICAgICAgICBsYXN0IGZ1c2VkIGZlYXQgLT4gY2hhbmdlX2hlYWQgLT4gY2hhbmdlX2xvZ2l0cwogICAgICAgIGxhc3QgZnVzZWQgZmVhdCAtPiBkaXNhc3Rlcl9oZWFkIC0+IGRpc2FzdGVyX2xvZ2l0cwoKICAgIFRoZSBlbmNvZGVyIGlzIFNIQVJFRCBiZXR3ZWVuIHByZSBhbmQgcG9zdCAodHJ1ZSBTaWFtZXNlLCBub3QgdHdpbikuCiAgICBUaGlzIHJlZHVjZXMgcGFyYW1ldGVycyBhbmQgZm9yY2VzIHRoZSBtb2RlbCB0byBsZWFybiBhIGNvbW1vbiByZXByZXNlbnRhdGlvbi4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIGJhY2tib25lX25hbWU9Im52aWRpYS9zZWdmb3JtZXItYjMtZmluZXR1bmVkLWFkZS01MTItNTEyIiwKICAgICAgICBudW1fZGFtYWdlX2NsYXNzZXM9NiwgICMgMD1iZyArIDUgZGFtYWdlIGxldmVscyAoaW5jbCB1bi1jbGFzc2lmaWVkKQogICAgICAgIG51bV9kaXNhc3Rlcl9jbGFzc2VzPTUsCiAgICAgICAgcHJldHJhaW5lZD1UcnVlLAogICAgKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKCiAgICAgICAgc2VsZi5udW1fZGFtYWdlX2NsYXNzZXMgPSBudW1fZGFtYWdlX2NsYXNzZXMKICAgICAgICBzZWxmLm51bV9kaXNhc3Rlcl9jbGFzc2VzID0gbnVtX2Rpc2FzdGVyX2NsYXNzZXMKCiAgICAgICAgIyA9PT0gMS4gTG9hZCBTZWdGb3JtZXItQjMgYmFja2JvbmUgPT09CiAgICAgICAgaWYgcHJldHJhaW5lZDoKICAgICAgICAgICAgYmFzZV9tb2RlbCA9IFNlZ2Zvcm1lckZvclNlbWFudGljU2VnbWVudGF0aW9uLmZyb21fcHJldHJhaW5lZCgKICAgICAgICAgICAgICAgIGJhY2tib25lX25hbWUsCiAgICAgICAgICAgICAgICBudW1fbGFiZWxzPW51bV9kYW1hZ2VfY2xhc3NlcywKICAgICAgICAgICAgICAgIGlnbm9yZV9taXNtYXRjaGVkX3NpemVzPVRydWUsCiAgICAgICAgICAgICkKICAgICAgICBlbHNlOgogICAgICAgICAgICBjb25maWcgPSBTZWdmb3JtZXJDb25maWcuZnJvbV9wcmV0cmFpbmVkKGJhY2tib25lX25hbWUpCiAgICAgICAgICAgIGNvbmZpZy5udW1fbGFiZWxzID0gbnVtX2RhbWFnZV9jbGFzc2VzCiAgICAgICAgICAgIGJhc2VfbW9kZWwgPSBTZWdmb3JtZXJGb3JTZW1hbnRpY1NlZ21lbnRhdGlvbihjb25maWcpCgogICAgICAgICMgU2hhcmVkIGVuY29kZXIgKGJvdGggcHJlIGFuZCBwb3N0IHdpbGwgdXNlIHRoaXMpCiAgICAgICAgc2VsZi5lbmNvZGVyID0gYmFzZV9tb2RlbC5zZWdmb3JtZXIuZW5jb2RlcgoKICAgICAgICAjIFNlZ0Zvcm1lciBkZWNvZGVyICh3ZSdsbCBmZWVkIGZ1c2VkIGZlYXR1cmVzIGludG8gaXQpCiAgICAgICAgc2VsZi5kZWNvZGVfaGVhZCA9IGJhc2VfbW9kZWwuZGVjb2RlX2hlYWQKCiAgICAgICAgIyA9PT0gMi4gRnVzaW9uIG1vZHVsZXMgYXQgZWFjaCBlbmNvZGVyIHN0YWdlID09PQogICAgICAgICMgU2VnRm9ybWVyLUIzIGhpZGRlbiBzaXplczogWzY0LCAxMjgsIDMyMCwgNTEyXQogICAgICAgIGVuY29kZXJfY2hhbm5lbHMgPSBiYXNlX21vZGVsLmNvbmZpZy5oaWRkZW5fc2l6ZXMgICMgWzY0LCAxMjgsIDMyMCwgNTEyXQogICAgICAgIHNlbGYuZW5jb2Rlcl9jaGFubmVscyA9IGVuY29kZXJfY2hhbm5lbHMKCiAgICAgICAgIyBGdXNpb246IGNvbmNhdChwcmUsIHBvc3QsIHxkaWZmfCkgLT4gMXgxIGNvbnYgLT4gb3JpZ2luYWwgY2hhbm5lbHMKICAgICAgICBzZWxmLmZ1c2lvbl9jb252cyA9IG5uLk1vZHVsZUxpc3QoWwogICAgICAgICAgICBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICAgICAgbm4uQ29udjJkKGNoICogMywgY2gsIGtlcm5lbF9zaXplPTEsIGJpYXM9RmFsc2UpLAogICAgICAgICAgICAgICAgbm4uQmF0Y2hOb3JtMmQoY2gpLAogICAgICAgICAgICAgICAgbm4uUmVMVShpbnBsYWNlPVRydWUpLAogICAgICAgICAgICApCiAgICAgICAgICAgIGZvciBjaCBpbiBlbmNvZGVyX2NoYW5uZWxzCiAgICAgICAgXSkKCiAgICAgICAgIyA9PT0gMy4gQ2hhbmdlIGhlYWQgKGJpbmFyeTogY2hhbmdlZC91bmNoYW5nZWQpID09PQogICAgICAgIGxhc3RfZGltID0gZW5jb2Rlcl9jaGFubmVsc1stMV0gICMgNTEyCiAgICAgICAgc2VsZi5jaGFuZ2VfaGVhZCA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIG5uLkNvbnYyZChsYXN0X2RpbSwgMjU2LCBrZXJuZWxfc2l6ZT0zLCBwYWRkaW5nPTEpLAogICAgICAgICAgICBubi5CYXRjaE5vcm0yZCgyNTYpLAogICAgICAgICAgICBubi5SZUxVKGlucGxhY2U9VHJ1ZSksCiAgICAgICAgICAgIG5uLkRyb3BvdXQyZCgwLjEpLAogICAgICAgICAgICBubi5Db252MmQoMjU2LCAyLCBrZXJuZWxfc2l6ZT0xKSwKICAgICAgICApCgogICAgICAgICMgPT09IDQuIERpc2FzdGVyIGNsYXNzaWZpY2F0aW9uIGhlYWQgPT09CiAgICAgICAgc2VsZi5kaXNhc3Rlcl9oZWFkID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgbm4uQWRhcHRpdmVBdmdQb29sMmQoMSksCiAgICAgICAgICAgIG5uLkZsYXR0ZW4oKSwKICAgICAgICAgICAgbm4uTGluZWFyKGxhc3RfZGltLCAyNTYpLAogICAgICAgICAgICBubi5SZUxVKGlucGxhY2U9VHJ1ZSksCiAgICAgICAgICAgIG5uLkRyb3BvdXQoMC4zKSwKICAgICAgICAgICAgbm4uTGluZWFyKDI1NiwgbnVtX2Rpc2FzdGVyX2NsYXNzZXMpLAogICAgICAgICkKCiAgICBkZWYgX2VuY29kZShzZWxmLCB4KToKICAgICAgICAjIFJ1biBTZWdGb3JtZXIgZW5jb2RlciwgcmV0dXJuIGxpc3Qgb2YgNCBzdGFnZSBvdXRwdXRzCiAgICAgICAgb3V0cHV0cyA9IHNlbGYuZW5jb2Rlcih4LCBvdXRwdXRfaGlkZGVuX3N0YXRlcz1UcnVlLCByZXR1cm5fZGljdD1UcnVlKQogICAgICAgIHJldHVybiBsaXN0KG91dHB1dHMuaGlkZGVuX3N0YXRlcykgICMgNCBmZWF0dXJlIG1hcHMgYXQgZGlmZmVyZW50IHNjYWxlcwoKICAgIGRlZiBfZnVzZV9mZWF0dXJlcyhzZWxmLCBmZWF0dXJlc19wcmUsIGZlYXR1cmVzX3Bvc3QpOgogICAgICAgICMgRnVzZSBwcmUgYW5kIHBvc3QgZmVhdHVyZXMgYXQgZWFjaCBzY2FsZToKICAgICAgICAjICAgICBmdXNlZFtpXSA9IGNvbnYoY29uY2F0KHByZVtpXSwgcG9zdFtpXSwgfHBvc3RbaV0gLSBwcmVbaV18KSkKICAgICAgICBmdXNlZCA9IFtdCiAgICAgICAgZm9yIGksIChwcmUsIHBvc3QpIGluIGVudW1lcmF0ZSh6aXAoZmVhdHVyZXNfcHJlLCBmZWF0dXJlc19wb3N0KSk6CiAgICAgICAgICAgIGRpZmYgPSB0b3JjaC5hYnMocG9zdCAtIHByZSkKICAgICAgICAgICAgY29tYmluZWQgPSB0b3JjaC5jYXQoW3ByZSwgcG9zdCwgZGlmZl0sIGRpbT0xKSAgIyBbQiwgM0MsIEgsIFddCiAgICAgICAgICAgIGZ1c2VkX2ZlYXQgPSBzZWxmLmZ1c2lvbl9jb252c1tpXShjb21iaW5lZCkKICAgICAgICAgICAgZnVzZWQuYXBwZW5kKGZ1c2VkX2ZlYXQpCiAgICAgICAgcmV0dXJuIGZ1c2VkCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgIyBBcmdzOgogICAgICAgICMgICAgIHg6IFtCLCA2LCBILCBXXSBjb25jYXQgW3ByZSwgcG9zdF0KICAgICAgICAjIFJldHVybnM6CiAgICAgICAgIyAgICAgZGljdCB3aXRoIGRhbWFnZV9sb2dpdHMsIGNoYW5nZV9sb2dpdHMsIGRpc2FzdGVyX2xvZ2l0cwogICAgICAgIEIsIEMsIEgsIFcgPSB4LnNoYXBlCiAgICAgICAgYXNzZXJ0IEMgPT0gNiwgZiJFeHBlY3RlZCA2IGNoYW5uZWxzIChwcmUrcG9zdCksIGdvdCB7Q30iCgogICAgICAgIHByZSA9IHhbOiwgOjNdICAgICMgW0IsIDMsIEgsIFddCiAgICAgICAgcG9zdCA9IHhbOiwgMzpdICAgIyBbQiwgMywgSCwgV10KCiAgICAgICAgIyBFbmNvZGUgYm90aCBicmFuY2hlcyAoU0hBUkVEIHdlaWdodHMpCiAgICAgICAgZmVhdHVyZXNfcHJlID0gc2VsZi5fZW5jb2RlKHByZSkKICAgICAgICBmZWF0dXJlc19wb3N0ID0gc2VsZi5fZW5jb2RlKHBvc3QpCgogICAgICAgICMgRnVzZSBhdCBlYWNoIHN0YWdlCiAgICAgICAgZnVzZWRfZmVhdHVyZXMgPSBzZWxmLl9mdXNlX2ZlYXR1cmVzKGZlYXR1cmVzX3ByZSwgZmVhdHVyZXNfcG9zdCkKCiAgICAgICAgIyBEYW1hZ2Ugc2VnbWVudGF0aW9uIHZpYSBTZWdGb3JtZXIgZGVjb2RlcgogICAgICAgIGRlY29kZXJfb3V0cHV0ID0gc2VsZi5kZWNvZGVfaGVhZChmdXNlZF9mZWF0dXJlcykKICAgICAgICBkYW1hZ2VfbG9naXRzID0gRi5pbnRlcnBvbGF0ZSgKICAgICAgICAgICAgZGVjb2Rlcl9vdXRwdXQsCiAgICAgICAgICAgIHNpemU9KEgsIFcpLAogICAgICAgICAgICBtb2RlPSJiaWxpbmVhciIsCiAgICAgICAgICAgIGFsaWduX2Nvcm5lcnM9RmFsc2UsCiAgICAgICAgKQoKICAgICAgICAjIENoYW5nZSBoZWFkOiB1c2UgdGhlIGxhc3QgZnVzZWQgZmVhdHVyZSAobG93ZXN0IHJlc29sdXRpb24pCiAgICAgICAgbGFzdF9mdXNlZCA9IGZ1c2VkX2ZlYXR1cmVzWy0xXSAgIyBbQiwgNTEyLCBILzMyLCBXLzMyXQogICAgICAgIGNoYW5nZV9sb3cgPSBzZWxmLmNoYW5nZV9oZWFkKGxhc3RfZnVzZWQpCiAgICAgICAgY2hhbmdlX2xvZ2l0cyA9IEYuaW50ZXJwb2xhdGUoCiAgICAgICAgICAgIGNoYW5nZV9sb3csCiAgICAgICAgICAgIHNpemU9KEgsIFcpLAogICAgICAgICAgICBtb2RlPSJiaWxpbmVhciIsCiAgICAgICAgICAgIGFsaWduX2Nvcm5lcnM9RmFsc2UsCiAgICAgICAgKQoKICAgICAgICAjIERpc2FzdGVyIGNsYXNzaWZpY2F0aW9uIGhlYWQKICAgICAgICBkaXNhc3Rlcl9sb2dpdHMgPSBzZWxmLmRpc2FzdGVyX2hlYWQobGFzdF9mdXNlZCkKCiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgImRhbWFnZV9sb2dpdHMiOiBkYW1hZ2VfbG9naXRzLAogICAgICAgICAgICAiY2hhbmdlX2xvZ2l0cyI6IGNoYW5nZV9sb2dpdHMsCiAgICAgICAgICAgICJkaXNhc3Rlcl9sb2dpdHMiOiBkaXNhc3Rlcl9sb2dpdHMsCiAgICAgICAgfQoKICAgIGRlZiBudW1fcGFyYW1ldGVycyhzZWxmKToKICAgICAgICByZXR1cm4gc3VtKHAubnVtZWwoKSBmb3IgcCBpbiBzZWxmLnBhcmFtZXRlcnMoKSBpZiBwLnJlcXVpcmVzX2dyYWQpCgogICAgZGVmIGVuYWJsZV9ncmFkaWVudF9jaGVja3BvaW50aW5nKHNlbGYpOgogICAgICAgICMgRW5hYmxlIGdyYWRpZW50IGNoZWNrcG9pbnRpbmcgb24gdGhlIGVuY29kZXIgdG8gc2F2ZSBtZW1vcnkuCiAgICAgICAgIyBUaGlzIHJlY29tcHV0ZXMgYWN0aXZhdGlvbnMgZHVyaW5nIGJhY2t3YXJkIHBhc3MgaW5zdGVhZCBvZiBzdG9yaW5nIHRoZW0uCiAgICAgICAgIyBSZWR1Y2VzIFZSQU0gYnkgfjMwJSBhdCB0aGUgY29zdCBvZiB+MTUlIHRyYWluaW5nIHRpbWUuCiAgICAgICAgaWYgaGFzYXR0cihzZWxmLmVuY29kZXIsICJncmFkaWVudF9jaGVja3BvaW50aW5nX2VuYWJsZSIpOgogICAgICAgICAgICBzZWxmLmVuY29kZXIuZ3JhZGllbnRfY2hlY2twb2ludGluZ19lbmFibGUoKQogICAgICAgICAgICByZXR1cm4gVHJ1ZQogICAgICAgICMgRmFsbGJhY2s6IG1hbnVhbGx5IHNldCB0aGUgZmxhZwogICAgICAgIGlmIGhhc2F0dHIoc2VsZi5lbmNvZGVyLCAiZ3JhZGllbnRfY2hlY2twb2ludGluZyIpOgogICAgICAgICAgICBzZWxmLmVuY29kZXIuZ3JhZGllbnRfY2hlY2twb2ludGluZyA9IFRydWUKICAgICAgICAgICAgcmV0dXJuIFRydWUKICAgICAgICByZXR1cm4gRmFsc2UKCgojID09PSBCYWNrd2FyZHMgY29tcGF0aWJpbGl0eToga2VlcCB0aGUgb2xkIGNsYXNzIG5hbWUgdG9vID09PQpjbGFzcyBUZWFjaGVyU2VnZm9ybWVyKFNpYW1lc2VUZWFjaGVyU2VnZm9ybWVyKToKICAgICMgQWxpYXMgZm9yIHYxIGNvbXBhdGliaWxpdHkKICAgIHBhc3MK").decode("utf-8")
LOSSES_V2 = base64.b64decode("IiIiCkFGRVRTT05BUiBMb3NzIEZ1bmN0aW9ucyB2MiDigJQgQ29tYm8gbG9zcyB3aXRoIEZvY2FsICsgRGljZSArIEJvdW5kYXJ5LgoKTXVsdGktdGFzayBsb3NzIGZvciB0ZWFjaGVyOgogICAgTF90b3RhbCA9IHdfZGFtYWdlICogTF9jb21ib19kYW1hZ2UgKyB3X2NoYW5nZSAqIExfY2VfY2hhbmdlICsgd19kaXNhc3RlciAqIExfY2VfZGlzYXN0ZXIKCiAgICBMX2NvbWJvX2RhbWFnZSA9IDAuNSAqIExfZm9jYWwgKyAwLjMgKiBMX2RpY2UgKyAwLjIgKiBMX2JvdW5kYXJ5CgpXaGVyZToKICAgIC0gTF9mb2NhbDogICAgZm9jdXNlcyBvbiBoYXJkIGV4YW1wbGVzIChjbGFzcyBpbWJhbGFuY2UpCiAgICAtIExfZGljZTogICAgIG1heGltaXplcyBwaXhlbCBvdmVybGFwIChib3VuZGFyeSBxdWFsaXR5KQogICAgLSBMX2JvdW5kYXJ5OiBleHRyYSBzaWduYWwgYXQgY2xhc3MgYm91bmRhcmllcyAoZmluZSBkZXRhaWwpCiIiIgoKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgppbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCgoKY2xhc3MgRm9jYWxMb3NzKG5uLk1vZHVsZSk6CiAgICAiIiJNdWx0aS1jbGFzcyBmb2NhbCBsb3NzIHdpdGggY2xhc3Mgd2VpZ2h0cy4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgZ2FtbWE9Mi4wLCBhbHBoYT1Ob25lLCBpZ25vcmVfaW5kZXg9LTEwMCk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5nYW1tYSA9IGdhbW1hCiAgICAgICAgc2VsZi5pZ25vcmVfaW5kZXggPSBpZ25vcmVfaW5kZXgKICAgICAgICBpZiBhbHBoYSBpcyBub3QgTm9uZToKICAgICAgICAgICAgc2VsZi5yZWdpc3Rlcl9idWZmZXIoImFscGhhIiwgdG9yY2gudGVuc29yKGFscGhhLCBkdHlwZT10b3JjaC5mbG9hdDMyKSkKICAgICAgICBlbHNlOgogICAgICAgICAgICBzZWxmLmFscGhhID0gTm9uZQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIGxvZ2l0cywgdGFyZ2V0cyk6CiAgICAgICAgY2VfbG9zcyA9IEYuY3Jvc3NfZW50cm9weSgKICAgICAgICAgICAgbG9naXRzLCB0YXJnZXRzLAogICAgICAgICAgICByZWR1Y3Rpb249Im5vbmUiLAogICAgICAgICAgICBpZ25vcmVfaW5kZXg9c2VsZi5pZ25vcmVfaW5kZXgsCiAgICAgICAgKQogICAgICAgIHBfdCA9IHRvcmNoLmV4cCgtY2VfbG9zcykKICAgICAgICBmb2NhbF93ZWlnaHQgPSAoMS4wIC0gcF90KSAqKiBzZWxmLmdhbW1hCgogICAgICAgIGlmIHNlbGYuYWxwaGEgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGFscGhhX3QgPSBzZWxmLmFscGhhW3RhcmdldHMuY2xhbXAobWluPTApXQogICAgICAgICAgICBmb2NhbF93ZWlnaHQgPSBmb2NhbF93ZWlnaHQgKiBhbHBoYV90CgogICAgICAgIGxvc3MgPSBmb2NhbF93ZWlnaHQgKiBjZV9sb3NzCiAgICAgICAgdmFsaWRfbWFzayA9ICh0YXJnZXRzICE9IHNlbGYuaWdub3JlX2luZGV4KS5mbG9hdCgpCiAgICAgICAgbG9zcyA9IGxvc3MgKiB2YWxpZF9tYXNrCgogICAgICAgIHJldHVybiBsb3NzLnN1bSgpIC8gdmFsaWRfbWFzay5zdW0oKS5jbGFtcChtaW49MS4wKQoKCmNsYXNzIERpY2VMb3NzKG5uLk1vZHVsZSk6CiAgICAiIiIKICAgIE11bHRpLWNsYXNzIERpY2UgTG9zcyBmb3Igc2VnbWVudGF0aW9uLgoKICAgIENvbXB1dGVzIERpY2UgY29lZmZpY2llbnQgcGVyIGNsYXNzIGFuZCByZXR1cm5zIDEgLSBtZWFuX2RpY2UuCiAgICBIYW5kbGVzIGNsYXNzIGltYmFsYW5jZSBiZXR0ZXIgdGhhbiBDRSBmb3Igc2VnbWVudGF0aW9uLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIG51bV9jbGFzc2VzLCBpZ25vcmVfaW5kZXg9LTEwMCwgc21vb3RoPTEuMCwKICAgICAgICAgICAgICAgICBjbGFzc193ZWlnaHRzPU5vbmUsIGV4Y2x1ZGVfYmFja2dyb3VuZD1UcnVlKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLm51bV9jbGFzc2VzID0gbnVtX2NsYXNzZXMKICAgICAgICBzZWxmLmlnbm9yZV9pbmRleCA9IGlnbm9yZV9pbmRleAogICAgICAgIHNlbGYuc21vb3RoID0gc21vb3RoCiAgICAgICAgc2VsZi5leGNsdWRlX2JhY2tncm91bmQgPSBleGNsdWRlX2JhY2tncm91bmQKICAgICAgICBpZiBjbGFzc193ZWlnaHRzIGlzIG5vdCBOb25lOgogICAgICAgICAgICBzZWxmLnJlZ2lzdGVyX2J1ZmZlcigiY2xhc3Nfd2VpZ2h0cyIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHRvcmNoLnRlbnNvcihjbGFzc193ZWlnaHRzLCBkdHlwZT10b3JjaC5mbG9hdDMyKSkKICAgICAgICBlbHNlOgogICAgICAgICAgICBzZWxmLmNsYXNzX3dlaWdodHMgPSBOb25lCgogICAgZGVmIGZvcndhcmQoc2VsZiwgbG9naXRzLCB0YXJnZXRzKToKICAgICAgICAjIGxvZ2l0czogIFtCLCBDLCBILCBXXQogICAgICAgICMgdGFyZ2V0czogW0IsIEgsIFddCiAgICAgICAgcHJvYnMgPSBGLnNvZnRtYXgobG9naXRzLCBkaW09MSkgICMgW0IsIEMsIEgsIFddCgogICAgICAgICMgT25lLWhvdCBlbmNvZGUgdGFyZ2V0cwogICAgICAgIHZhbGlkX21hc2sgPSAodGFyZ2V0cyAhPSBzZWxmLmlnbm9yZV9pbmRleCkKICAgICAgICB0YXJnZXRzX2NsYW1wZWQgPSB0YXJnZXRzLmNsYW1wKG1pbj0wLCBtYXg9c2VsZi5udW1fY2xhc3NlcyAtIDEpCiAgICAgICAgdGFyZ2V0c19vbmVob3QgPSBGLm9uZV9ob3QodGFyZ2V0c19jbGFtcGVkLCBudW1fY2xhc3Nlcz1zZWxmLm51bV9jbGFzc2VzKQogICAgICAgIHRhcmdldHNfb25laG90ID0gdGFyZ2V0c19vbmVob3QucGVybXV0ZSgwLCAzLCAxLCAyKS5mbG9hdCgpICAjIFtCLCBDLCBILCBXXQoKICAgICAgICAjIEFwcGx5IHZhbGlkIG1hc2sKICAgICAgICB2YWxpZF9tYXNrX2MgPSB2YWxpZF9tYXNrLnVuc3F1ZWV6ZSgxKS5mbG9hdCgpCiAgICAgICAgcHJvYnMgPSBwcm9icyAqIHZhbGlkX21hc2tfYwogICAgICAgIHRhcmdldHNfb25laG90ID0gdGFyZ2V0c19vbmVob3QgKiB2YWxpZF9tYXNrX2MKCiAgICAgICAgIyBEaWNlIHBlciBjbGFzcyAoYWNyb3NzIEIsIEgsIFcpCiAgICAgICAgZGltcyA9ICgwLCAyLCAzKQogICAgICAgIGludGVyc2VjdGlvbiA9IChwcm9icyAqIHRhcmdldHNfb25laG90KS5zdW0oZGltcykKICAgICAgICBjYXJkaW5hbGl0eSA9IHByb2JzLnN1bShkaW1zKSArIHRhcmdldHNfb25laG90LnN1bShkaW1zKQogICAgICAgIGRpY2UgPSAoMi4wICogaW50ZXJzZWN0aW9uICsgc2VsZi5zbW9vdGgpIC8gKGNhcmRpbmFsaXR5ICsgc2VsZi5zbW9vdGgpCgogICAgICAgICMgRXhjbHVkZSBiYWNrZ3JvdW5kIGlmIHJlcXVlc3RlZAogICAgICAgIHN0YXJ0X2lkeCA9IDEgaWYgc2VsZi5leGNsdWRlX2JhY2tncm91bmQgZWxzZSAwCgogICAgICAgIGlmIHNlbGYuY2xhc3Nfd2VpZ2h0cyBpcyBub3QgTm9uZToKICAgICAgICAgICAgd2VpZ2h0cyA9IHNlbGYuY2xhc3Nfd2VpZ2h0c1tzdGFydF9pZHg6XQogICAgICAgICAgICBkaWNlX2xvc3MgPSAoMS4wIC0gZGljZVtzdGFydF9pZHg6XSkgKiB3ZWlnaHRzCiAgICAgICAgICAgIHJldHVybiBkaWNlX2xvc3Muc3VtKCkgLyB3ZWlnaHRzLnN1bSgpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgcmV0dXJuICgxLjAgLSBkaWNlW3N0YXJ0X2lkeDpdKS5tZWFuKCkKCgpjbGFzcyBCb3VuZGFyeUxvc3Mobm4uTW9kdWxlKToKICAgICIiIgogICAgQm91bmRhcnktYXdhcmUgbG9zcy4KCiAgICBDb21wdXRlcyBDRSBsb3NzIHdpdGggZXh0cmEgd2VpZ2h0IGF0IGNsYXNzIGJvdW5kYXJpZXMgKHBpeGVscyBuZWFyCiAgICB0cmFuc2l0aW9ucykuIFRoaXMgaGVscHMgdGhlIG1vZGVsIGxlYXJuIHByZWNpc2UgYnVpbGRpbmcgZWRnZXMuCgogICAgSW1wbGVtZW50YXRpb246IGNvbXB1dGUgYSBkaXN0YW5jZS13ZWlnaHRlZCB2ZXJzaW9uIG9mIENFLCB3aGVyZQogICAgcGl4ZWxzIGNsb3NlIHRvIGNsYXNzIHRyYW5zaXRpb25zIGdldCBoaWdoZXIgd2VpZ2h0LgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIG51bV9jbGFzc2VzLCBrZXJuZWxfc2l6ZT01LCBpZ25vcmVfaW5kZXg9LTEwMCk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5udW1fY2xhc3NlcyA9IG51bV9jbGFzc2VzCiAgICAgICAgc2VsZi5rZXJuZWxfc2l6ZSA9IGtlcm5lbF9zaXplCiAgICAgICAgc2VsZi5pZ25vcmVfaW5kZXggPSBpZ25vcmVfaW5kZXgKCiAgICBAdG9yY2gubm9fZ3JhZCgpCiAgICBkZWYgX2NvbXB1dGVfYm91bmRhcnlfd2VpZ2h0cyhzZWxmLCB0YXJnZXRzKToKICAgICAgICAjIEZpbmQgcGl4ZWxzIGF0IGNsYXNzIGJvdW5kYXJpZXMgdXNpbmcgbWF4LXBvb2wgZGlsYXRpb24gdHJpY2suCiAgICAgICAgIyBCb3VuZGFyeSBwaXhlbHMgZ2V0IHdlaWdodCAyLjAsIGludGVyaW9yIHBpeGVscyBnZXQgMS4wLgogICAgICAgIEIsIEgsIFcgPSB0YXJnZXRzLnNoYXBlCiAgICAgICAgdGFyZ2V0c19tYXNrID0gKHRhcmdldHMgIT0gc2VsZi5pZ25vcmVfaW5kZXgpLmZsb2F0KCkKICAgICAgICBzYWZlX3RhcmdldHMgPSB0YXJnZXRzLmNsYW1wKG1pbj0wKS5mbG9hdCgpLnVuc3F1ZWV6ZSgxKSAgIyBbQiwxLEgsV10KCiAgICAgICAgIyBEaWxhdGVkIHZlcnNpb246IGlmIGFueSBuZWlnaGJvciBoYXMgYSBkaWZmZXJlbnQgY2xhc3MgLT4gYm91bmRhcnkKICAgICAgICBwYWQgPSBzZWxmLmtlcm5lbF9zaXplIC8vIDIKICAgICAgICBtYXhfcG9vbCA9IEYubWF4X3Bvb2wyZChzYWZlX3RhcmdldHMsIHNlbGYua2VybmVsX3NpemUsIHN0cmlkZT0xLCBwYWRkaW5nPXBhZCkKICAgICAgICBtaW5fcG9vbCA9IC1GLm1heF9wb29sMmQoLXNhZmVfdGFyZ2V0cywgc2VsZi5rZXJuZWxfc2l6ZSwgc3RyaWRlPTEsIHBhZGRpbmc9cGFkKQogICAgICAgIGlzX2JvdW5kYXJ5ID0gKG1heF9wb29sICE9IG1pbl9wb29sKS5mbG9hdCgpLnNxdWVlemUoMSkgICMgW0IsSCxXXQoKICAgICAgICAjIFdlaWdodDogMi4wIG9uIGJvdW5kYXJ5LCAxLjAgZWxzZXdoZXJlCiAgICAgICAgd2VpZ2h0cyA9IDEuMCArIGlzX2JvdW5kYXJ5CiAgICAgICAgd2VpZ2h0cyA9IHdlaWdodHMgKiB0YXJnZXRzX21hc2sKICAgICAgICByZXR1cm4gd2VpZ2h0cwoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIGxvZ2l0cywgdGFyZ2V0cyk6CiAgICAgICAgY2UgPSBGLmNyb3NzX2VudHJvcHkoCiAgICAgICAgICAgIGxvZ2l0cywgdGFyZ2V0cywKICAgICAgICAgICAgcmVkdWN0aW9uPSJub25lIiwKICAgICAgICAgICAgaWdub3JlX2luZGV4PXNlbGYuaWdub3JlX2luZGV4LAogICAgICAgICkgICMgW0IsIEgsIFddCgogICAgICAgIHdlaWdodHMgPSBzZWxmLl9jb21wdXRlX2JvdW5kYXJ5X3dlaWdodHModGFyZ2V0cykKICAgICAgICBsb3NzID0gKGNlICogd2VpZ2h0cykuc3VtKCkgLyB3ZWlnaHRzLnN1bSgpLmNsYW1wKG1pbj0xLjApCiAgICAgICAgcmV0dXJuIGxvc3MKCgpjbGFzcyBDb21ib0RhbWFnZUxvc3Mobm4uTW9kdWxlKToKICAgICIiIgogICAgQ29tYmluZWQgbG9zcyBmb3IgZGFtYWdlIHNlZ21lbnRhdGlvbjoKICAgICAgICAwLjUgKiBGb2NhbCArIDAuMyAqIERpY2UgKyAwLjIgKiBCb3VuZGFyeQogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgbnVtX2NsYXNzZXMsCiAgICAgICAgY2xhc3Nfd2VpZ2h0cz1Ob25lLAogICAgICAgIGZvY2FsX2dhbW1hPTIuMCwKICAgICAgICBpZ25vcmVfaW5kZXg9LTEwMCwKICAgICAgICBmb2NhbF93ZWlnaHQ9MC41LAogICAgICAgIGRpY2Vfd2VpZ2h0PTAuMywKICAgICAgICBib3VuZGFyeV93ZWlnaHQ9MC4yLAogICAgKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmZvY2FsID0gRm9jYWxMb3NzKAogICAgICAgICAgICBnYW1tYT1mb2NhbF9nYW1tYSwKICAgICAgICAgICAgYWxwaGE9Y2xhc3Nfd2VpZ2h0cywKICAgICAgICAgICAgaWdub3JlX2luZGV4PWlnbm9yZV9pbmRleCwKICAgICAgICApCiAgICAgICAgc2VsZi5kaWNlID0gRGljZUxvc3MoCiAgICAgICAgICAgIG51bV9jbGFzc2VzPW51bV9jbGFzc2VzLAogICAgICAgICAgICBjbGFzc193ZWlnaHRzPWNsYXNzX3dlaWdodHMsCiAgICAgICAgICAgIGlnbm9yZV9pbmRleD1pZ25vcmVfaW5kZXgsCiAgICAgICAgICAgIGV4Y2x1ZGVfYmFja2dyb3VuZD1UcnVlLAogICAgICAgICkKICAgICAgICBzZWxmLmJvdW5kYXJ5ID0gQm91bmRhcnlMb3NzKAogICAgICAgICAgICBudW1fY2xhc3Nlcz1udW1fY2xhc3NlcywKICAgICAgICAgICAgaWdub3JlX2luZGV4PWlnbm9yZV9pbmRleCwKICAgICAgICApCiAgICAgICAgc2VsZi5mb2NhbF93ZWlnaHQgPSBmb2NhbF93ZWlnaHQKICAgICAgICBzZWxmLmRpY2Vfd2VpZ2h0ID0gZGljZV93ZWlnaHQKICAgICAgICBzZWxmLmJvdW5kYXJ5X3dlaWdodCA9IGJvdW5kYXJ5X3dlaWdodAoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIGxvZ2l0cywgdGFyZ2V0cyk6CiAgICAgICAgZm9jYWxfbG9zcyA9IHNlbGYuZm9jYWwobG9naXRzLCB0YXJnZXRzKQogICAgICAgIGRpY2VfbG9zcyA9IHNlbGYuZGljZShsb2dpdHMsIHRhcmdldHMpCiAgICAgICAgYm91bmRhcnlfbG9zcyA9IHNlbGYuYm91bmRhcnkobG9naXRzLCB0YXJnZXRzKQoKICAgICAgICB0b3RhbCA9ICgKICAgICAgICAgICAgc2VsZi5mb2NhbF93ZWlnaHQgKiBmb2NhbF9sb3NzCiAgICAgICAgICAgICsgc2VsZi5kaWNlX3dlaWdodCAqIGRpY2VfbG9zcwogICAgICAgICAgICArIHNlbGYuYm91bmRhcnlfd2VpZ2h0ICogYm91bmRhcnlfbG9zcwogICAgICAgICkKCiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgInRvdGFsIjogdG90YWwsCiAgICAgICAgICAgICJmb2NhbCI6IGZvY2FsX2xvc3MuZGV0YWNoKCksCiAgICAgICAgICAgICJkaWNlIjogZGljZV9sb3NzLmRldGFjaCgpLAogICAgICAgICAgICAiYm91bmRhcnkiOiBib3VuZGFyeV9sb3NzLmRldGFjaCgpLAogICAgICAgIH0KCgpjbGFzcyBUZWFjaGVyTG9zc1YyKG5uLk1vZHVsZSk6CiAgICAiIiIKICAgIFBsYW4gdjIgbXVsdGktdGFzayB0ZWFjaGVyIGxvc3M6CiAgICAgICAgTCA9IHdfZG1nICogTF9jb21ib19kYW1hZ2UgKyB3X2NoZyAqIExfY2VfY2hhbmdlICsgd19kaXMgKiBMX2NlX2Rpc2FzdGVyCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICBudW1fZGFtYWdlX2NsYXNzZXM9NiwKICAgICAgICBkYW1hZ2Vfd2VpZ2h0PTAuNzAsCiAgICAgICAgY2hhbmdlX3dlaWdodD0wLjIwLAogICAgICAgIGRpc2FzdGVyX3dlaWdodD0wLjEwLAogICAgICAgIGZvY2FsX2dhbW1hPTIuMCwKICAgICAgICBkYW1hZ2VfY2xhc3Nfd2VpZ2h0cz1Ob25lLAogICAgKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmRhbWFnZV93ZWlnaHQgPSBkYW1hZ2Vfd2VpZ2h0CiAgICAgICAgc2VsZi5jaGFuZ2Vfd2VpZ2h0ID0gY2hhbmdlX3dlaWdodAogICAgICAgIHNlbGYuZGlzYXN0ZXJfd2VpZ2h0ID0gZGlzYXN0ZXJfd2VpZ2h0CgogICAgICAgIHNlbGYuY29tYm9fZGFtYWdlID0gQ29tYm9EYW1hZ2VMb3NzKAogICAgICAgICAgICBudW1fY2xhc3Nlcz1udW1fZGFtYWdlX2NsYXNzZXMsCiAgICAgICAgICAgIGNsYXNzX3dlaWdodHM9ZGFtYWdlX2NsYXNzX3dlaWdodHMsCiAgICAgICAgICAgIGZvY2FsX2dhbW1hPWZvY2FsX2dhbW1hLAogICAgICAgICkKICAgICAgICBzZWxmLmNlX2NoYW5nZSA9IG5uLkNyb3NzRW50cm9weUxvc3MoKQogICAgICAgIHNlbGYuY2VfZGlzYXN0ZXIgPSBubi5Dcm9zc0VudHJvcHlMb3NzKCkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBvdXRwdXRzLCB0YXJnZXRzKToKICAgICAgICBkYW1hZ2VfbG9zc2VzID0gc2VsZi5jb21ib19kYW1hZ2Uob3V0cHV0c1siZGFtYWdlX2xvZ2l0cyJdLCB0YXJnZXRzWyJkYW1hZ2VfbWFzayJdKQogICAgICAgIGNoYW5nZV9sb3NzID0gc2VsZi5jZV9jaGFuZ2Uob3V0cHV0c1siY2hhbmdlX2xvZ2l0cyJdLCB0YXJnZXRzWyJjaGFuZ2VfbWFzayJdKQogICAgICAgIGRpc2FzdGVyX2xvc3MgPSBzZWxmLmNlX2Rpc2FzdGVyKG91dHB1dHNbImRpc2FzdGVyX2xvZ2l0cyJdLCB0YXJnZXRzWyJkaXNhc3Rlcl9pZHgiXSkKCiAgICAgICAgdG90YWwgPSAoCiAgICAgICAgICAgIHNlbGYuZGFtYWdlX3dlaWdodCAqIGRhbWFnZV9sb3NzZXNbInRvdGFsIl0KICAgICAgICAgICAgKyBzZWxmLmNoYW5nZV93ZWlnaHQgKiBjaGFuZ2VfbG9zcwogICAgICAgICAgICArIHNlbGYuZGlzYXN0ZXJfd2VpZ2h0ICogZGlzYXN0ZXJfbG9zcwogICAgICAgICkKCiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgInRvdGFsIjogdG90YWwsCiAgICAgICAgICAgICJkYW1hZ2UiOiBkYW1hZ2VfbG9zc2VzWyJ0b3RhbCJdLmRldGFjaCgpLAogICAgICAgICAgICAiZGFtYWdlX2ZvY2FsIjogZGFtYWdlX2xvc3Nlc1siZm9jYWwiXSwKICAgICAgICAgICAgImRhbWFnZV9kaWNlIjogZGFtYWdlX2xvc3Nlc1siZGljZSJdLAogICAgICAgICAgICAiZGFtYWdlX2JvdW5kYXJ5IjogZGFtYWdlX2xvc3Nlc1siYm91bmRhcnkiXSwKICAgICAgICAgICAgImNoYW5nZSI6IGNoYW5nZV9sb3NzLmRldGFjaCgpLAogICAgICAgICAgICAiZGlzYXN0ZXIiOiBkaXNhc3Rlcl9sb3NzLmRldGFjaCgpLAogICAgICAgIH0KCgpkZWYgZGVyaXZlX2NoYW5nZV9tYXNrX3YyKGRhbWFnZV9tYXNrKToKICAgICIiIgogICAgRGVyaXZlIGJpbmFyeSBjaGFuZ2UgbWFzayBmcm9tIGRhbWFnZSBtYXNrICh2MiB3aXRoIDYgY2xhc3NlcykuCgogICAgQ2hhbmdlID0gZGFtYWdlZCBidWlsZGluZzogbWlub3IoMiksIG1ham9yKDMpLCBkZXN0cm95ZWQoNCkuCiAgICBVbmNoYW5nZWQgPSBiYWNrZ3JvdW5kKDApLCBub19kYW1hZ2UoMSksIHVuLWNsYXNzaWZpZWQoNSkKICAgICh1bi1jbGFzc2lmaWVkIHRyZWF0ZWQgYXMgIm5vIGNlcnRhaW50eSBvZiBjaGFuZ2UiKQogICAgIiIiCiAgICByZXR1cm4gKChkYW1hZ2VfbWFzayA+PSAyKSAmIChkYW1hZ2VfbWFzayA8PSA0KSkubG9uZygpCg==").decode("utf-8")
DATASET_V2 = base64.b64decode("IiIiCkFGRVRTT05BUiBEYXRhc2V0IHYyIOKAlCBCdWlsZGluZy1hd2FyZSBjcm9wLCA2IGNsYXNzIHN1cHBvcnQuCgpLZXkgZGlmZmVyZW5jZXMgZnJvbSB2MToKLSA2IGRhbWFnZSBjbGFzc2VzIChpbmNsdWRlcyB1bi1jbGFzc2lmaWVkID0gNSkKLSBCdWlsZGluZy1hd2FyZSBjcm9wOiA4MCUgb2YgdGltZSwgY3JvcCBpcyBjZW50ZXJlZCBvbiBhIGJ1aWxkaW5nIHJlZ2lvbgotIFJldHVybnMgZGljdCB3aXRoIHNhbXBsZV93ZWlnaHQgZm9yIFdlaWdodGVkUmFuZG9tU2FtcGxlcgotIFN1cHBvcnRzIG1vZGU9J3RlYWNoZXInICg2IGNoKSBvciBtb2RlPSdzdHVkZW50JyAoMyBjaCkKIiIiCgppbXBvcnQgb3MKaW1wb3J0IGN2MgppbXBvcnQgcmFuZG9tCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFzZXQKCgpjbGFzcyBYQkREYXRhc2V0VjIoRGF0YXNldCk6CiAgICAiIiIKICAgIHhCRCBkYW1hZ2Ugc2VnbWVudGF0aW9uIGRhdGFzZXQgdjIuCgogICAgUmV0dXJucyBkaWN0OgogICAgICAgIC0gaW1hZ2U6ICAgICAgICBUZW5zb3IgW0MsIEgsIFddCiAgICAgICAgLSBtYXNrOiAgICAgICAgIFRlbnNvciBbSCwgV10gICAobG9uZywgMC4uNSkKICAgICAgICAtIGRpc2FzdGVyX2lkeDogVGVuc29yIHNjYWxhciAgIChsb25nLCAwLi40KQogICAgICAgIC0gZmlsZW5hbWU6ICAgICBzdHIKICAgICIiIgoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIGNzdl9wYXRoLAogICAgICAgIG1vZGU9InRlYWNoZXIiLAogICAgICAgIGF1Z21lbnRhdGlvbj1Ob25lLAogICAgICAgIGltYWdlX3NpemU9NTEyLAogICAgICAgIGJ1aWxkaW5nX2F3YXJlX2Nyb3A9VHJ1ZSwKICAgICAgICBidWlsZGluZ19jcm9wX3Byb2I9MC44LAogICAgKToKICAgICAgICBhc3NlcnQgbW9kZSBpbiAoInRlYWNoZXIiLCAic3R1ZGVudCIpCiAgICAgICAgc2VsZi5kZiA9IHBkLnJlYWRfY3N2KGNzdl9wYXRoKQogICAgICAgIHNlbGYubW9kZSA9IG1vZGUKICAgICAgICBzZWxmLmF1Z21lbnRhdGlvbiA9IGF1Z21lbnRhdGlvbgogICAgICAgIHNlbGYuaW1hZ2Vfc2l6ZSA9IGltYWdlX3NpemUKICAgICAgICBzZWxmLmJ1aWxkaW5nX2F3YXJlX2Nyb3AgPSBidWlsZGluZ19hd2FyZV9jcm9wCiAgICAgICAgc2VsZi5idWlsZGluZ19jcm9wX3Byb2IgPSBidWlsZGluZ19jcm9wX3Byb2IKCiAgICBkZWYgX19sZW5fXyhzZWxmKToKICAgICAgICByZXR1cm4gbGVuKHNlbGYuZGYpCgogICAgZGVmIF9sb2FkX2ltYWdlKHNlbGYsIHBhdGgpOgogICAgICAgIGltZyA9IGN2Mi5pbXJlYWQocGF0aCkKICAgICAgICBpZiBpbWcgaXMgTm9uZToKICAgICAgICAgICAgcmV0dXJuIG5wLnplcm9zKChzZWxmLmltYWdlX3NpemUsIHNlbGYuaW1hZ2Vfc2l6ZSwgMyksIGR0eXBlPW5wLnVpbnQ4KQogICAgICAgIHJldHVybiBjdjIuY3Z0Q29sb3IoaW1nLCBjdjIuQ09MT1JfQkdSMlJHQikKCiAgICBkZWYgX2xvYWRfbWFzayhzZWxmLCBwYXRoKToKICAgICAgICBtYXNrID0gY3YyLmltcmVhZChwYXRoLCBjdjIuSU1SRUFEX0dSQVlTQ0FMRSkKICAgICAgICBpZiBtYXNrIGlzIE5vbmU6CiAgICAgICAgICAgIHJldHVybiBucC56ZXJvcygoc2VsZi5pbWFnZV9zaXplLCBzZWxmLmltYWdlX3NpemUpLCBkdHlwZT1ucC51aW50OCkKICAgICAgICByZXR1cm4gbWFzawoKICAgIGRlZiBfYnVpbGRpbmdfYXdhcmVfY3JvcChzZWxmLCBwb3N0LCBtYXNrLCBwcmU9Tm9uZSk6CiAgICAgICAgIyBDcm9wIHRoYXQgcHJlZmVycyByZWdpb25zIGNvbnRhaW5pbmcgYnVpbGRpbmdzIChub24temVybyBtYXNrKS4KICAgICAgICAjIFJldHVybnMgY3JvcHBlZCBwb3N0LCBtYXNrLCAoYW5kIHByZSBpZiBwcm92aWRlZCkuCiAgICAgICAgaCwgdyA9IG1hc2suc2hhcGUKICAgICAgICB0YXJnZXRfaCA9IHRhcmdldF93ID0gc2VsZi5pbWFnZV9zaXplCgogICAgICAgICMgSWYgYWxyZWFkeSBzbWFsbGVyLCByZXR1cm4gYXMtaXMKICAgICAgICBpZiBoIDw9IHRhcmdldF9oIGFuZCB3IDw9IHRhcmdldF93OgogICAgICAgICAgICByZXR1cm4gcG9zdCwgbWFzaywgcHJlCgogICAgICAgICMgRmluZCBidWlsZGluZyBiYm94IChpZiBhbnkpCiAgICAgICAgbm9uemVybyA9IG5wLndoZXJlKG1hc2sgPiAwKQogICAgICAgIGlmIGxlbihub256ZXJvWzBdKSA9PSAwIG9yIHJhbmRvbS5yYW5kb20oKSA+IHNlbGYuYnVpbGRpbmdfY3JvcF9wcm9iOgogICAgICAgICAgICAjIE5vIGJ1aWxkaW5ncyBPUiByYW5kb20gY3JvcAogICAgICAgICAgICB5X3N0YXJ0ID0gcmFuZG9tLnJhbmRpbnQoMCwgbWF4KDAsIGggLSB0YXJnZXRfaCkpCiAgICAgICAgICAgIHhfc3RhcnQgPSByYW5kb20ucmFuZGludCgwLCBtYXgoMCwgdyAtIHRhcmdldF93KSkKICAgICAgICBlbHNlOgogICAgICAgICAgICAjIFBpY2sgYSByYW5kb20gYnVpbGRpbmcgcGl4ZWwgYW5kIGNyb3AgYXJvdW5kIGl0CiAgICAgICAgICAgIGlkeCA9IHJhbmRvbS5yYW5kaW50KDAsIGxlbihub256ZXJvWzBdKSAtIDEpCiAgICAgICAgICAgIGN5LCBjeCA9IG5vbnplcm9bMF1baWR4XSwgbm9uemVyb1sxXVtpZHhdCgogICAgICAgICAgICAjIE9mZnNldCBzbyB0aGUgYnVpbGRpbmcgaXMgcm91Z2hseSBjZW50ZXJlZCBidXQgd2l0aCBzb21lIGppdHRlcgogICAgICAgICAgICBqaXR0ZXJfaCA9IHRhcmdldF9oIC8vIDQKICAgICAgICAgICAgaml0dGVyX3cgPSB0YXJnZXRfdyAvLyA0CiAgICAgICAgICAgIHlfc3RhcnQgPSBjeSAtIHRhcmdldF9oIC8vIDIgKyByYW5kb20ucmFuZGludCgtaml0dGVyX2gsIGppdHRlcl9oKQogICAgICAgICAgICB4X3N0YXJ0ID0gY3ggLSB0YXJnZXRfdyAvLyAyICsgcmFuZG9tLnJhbmRpbnQoLWppdHRlcl93LCBqaXR0ZXJfdykKCiAgICAgICAgICAgICMgQ2xhbXAgdG8gdmFsaWQgcmFuZ2UKICAgICAgICAgICAgeV9zdGFydCA9IG1heCgwLCBtaW4oaCAtIHRhcmdldF9oLCB5X3N0YXJ0KSkKICAgICAgICAgICAgeF9zdGFydCA9IG1heCgwLCBtaW4odyAtIHRhcmdldF93LCB4X3N0YXJ0KSkKCiAgICAgICAgeV9lbmQgPSB5X3N0YXJ0ICsgdGFyZ2V0X2gKICAgICAgICB4X2VuZCA9IHhfc3RhcnQgKyB0YXJnZXRfdwoKICAgICAgICBwb3N0X2Nyb3AgPSBwb3N0W3lfc3RhcnQ6eV9lbmQsIHhfc3RhcnQ6eF9lbmRdCiAgICAgICAgbWFza19jcm9wID0gbWFza1t5X3N0YXJ0OnlfZW5kLCB4X3N0YXJ0OnhfZW5kXQogICAgICAgIHByZV9jcm9wID0gcHJlW3lfc3RhcnQ6eV9lbmQsIHhfc3RhcnQ6eF9lbmRdIGlmIHByZSBpcyBub3QgTm9uZSBlbHNlIE5vbmUKCiAgICAgICAgcmV0dXJuIHBvc3RfY3JvcCwgbWFza19jcm9wLCBwcmVfY3JvcAoKICAgIGRlZiBfX2dldGl0ZW1fXyhzZWxmLCBpZHgpOgogICAgICAgIHJvdyA9IHNlbGYuZGYuaWxvY1tpZHhdCgogICAgICAgIHBvc3QgPSBzZWxmLl9sb2FkX2ltYWdlKHJvd1sicG9zdF9wYXRoIl0pCiAgICAgICAgbWFzayA9IHNlbGYuX2xvYWRfbWFzayhyb3dbIm1hc2tfcGF0aCJdKQoKICAgICAgICAjIFNpemUgbWF0Y2gKICAgICAgICBpZiBwb3N0LnNoYXBlWzoyXSAhPSBtYXNrLnNoYXBlWzoyXToKICAgICAgICAgICAgbWFzayA9IGN2Mi5yZXNpemUoCiAgICAgICAgICAgICAgICBtYXNrLCAocG9zdC5zaGFwZVsxXSwgcG9zdC5zaGFwZVswXSksCiAgICAgICAgICAgICAgICBpbnRlcnBvbGF0aW9uPWN2Mi5JTlRFUl9ORUFSRVNULAogICAgICAgICAgICApCgogICAgICAgIGlmIHNlbGYubW9kZSA9PSAidGVhY2hlciI6CiAgICAgICAgICAgIHByZSA9IHNlbGYuX2xvYWRfaW1hZ2Uocm93WyJwcmVfcGF0aCJdKQogICAgICAgICAgICBpZiBwcmUuc2hhcGVbOjJdICE9IHBvc3Quc2hhcGVbOjJdOgogICAgICAgICAgICAgICAgcHJlID0gY3YyLnJlc2l6ZShwcmUsIChwb3N0LnNoYXBlWzFdLCBwb3N0LnNoYXBlWzBdKSkKCiAgICAgICAgICAgICMgQnVpbGRpbmctYXdhcmUgY3JvcCAob25seSBpZiBpbWFnZSBpcyBsYXJnZXIgdGhhbiB0YXJnZXQpCiAgICAgICAgICAgIGlmIHNlbGYuYnVpbGRpbmdfYXdhcmVfY3JvcCBhbmQgcG9zdC5zaGFwZVswXSA+IHNlbGYuaW1hZ2Vfc2l6ZToKICAgICAgICAgICAgICAgIHBvc3QsIG1hc2ssIHByZSA9IHNlbGYuX2J1aWxkaW5nX2F3YXJlX2Nyb3AocG9zdCwgbWFzaywgcHJlKQoKICAgICAgICAgICAgIyBBdWdtZW50YXRpb24gKHN5bmNlZCB2aWEgYWRkaXRpb25hbF90YXJnZXRzKQogICAgICAgICAgICBpZiBzZWxmLmF1Z21lbnRhdGlvbiBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIGF1Z21lbnRlZCA9IHNlbGYuYXVnbWVudGF0aW9uKGltYWdlPXBvc3QsIHByZT1wcmUsIG1hc2s9bWFzaykKICAgICAgICAgICAgICAgIHBvc3QgPSBhdWdtZW50ZWRbImltYWdlIl0KICAgICAgICAgICAgICAgIHByZSA9IGF1Z21lbnRlZFsicHJlIl0KICAgICAgICAgICAgICAgIG1hc2sgPSBhdWdtZW50ZWRbIm1hc2siXQoKICAgICAgICAgICAgIyBCdWlsZCA2LWNoYW5uZWwgaW5wdXQgW3ByZSwgcG9zdF0KICAgICAgICAgICAgaWYgaXNpbnN0YW5jZShwb3N0LCB0b3JjaC5UZW5zb3IpOgogICAgICAgICAgICAgICAgaW1hZ2UgPSB0b3JjaC5jYXQoW3ByZSwgcG9zdF0sIGRpbT0wKQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgaW1hZ2UgPSBucC5jb25jYXRlbmF0ZShbcHJlLCBwb3N0XSwgYXhpcz0yKQogICAgICAgICAgICAgICAgaW1hZ2UgPSB0b3JjaC5mcm9tX251bXB5KGltYWdlKS5wZXJtdXRlKDIsIDAsIDEpLmZsb2F0KCkgLyAyNTUuMAogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGlmIHNlbGYuYnVpbGRpbmdfYXdhcmVfY3JvcCBhbmQgcG9zdC5zaGFwZVswXSA+IHNlbGYuaW1hZ2Vfc2l6ZToKICAgICAgICAgICAgICAgIHBvc3QsIG1hc2ssIF8gPSBzZWxmLl9idWlsZGluZ19hd2FyZV9jcm9wKHBvc3QsIG1hc2ssIE5vbmUpCgogICAgICAgICAgICBpZiBzZWxmLmF1Z21lbnRhdGlvbiBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIGF1Z21lbnRlZCA9IHNlbGYuYXVnbWVudGF0aW9uKGltYWdlPXBvc3QsIG1hc2s9bWFzaykKICAgICAgICAgICAgICAgIHBvc3QgPSBhdWdtZW50ZWRbImltYWdlIl0KICAgICAgICAgICAgICAgIG1hc2sgPSBhdWdtZW50ZWRbIm1hc2siXQoKICAgICAgICAgICAgaWYgaXNpbnN0YW5jZShwb3N0LCB0b3JjaC5UZW5zb3IpOgogICAgICAgICAgICAgICAgaW1hZ2UgPSBwb3N0CiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBpbWFnZSA9IHRvcmNoLmZyb21fbnVtcHkocG9zdCkucGVybXV0ZSgyLCAwLCAxKS5mbG9hdCgpIC8gMjU1LjAKCiAgICAgICAgaWYgbm90IGlzaW5zdGFuY2UobWFzaywgdG9yY2guVGVuc29yKToKICAgICAgICAgICAgbWFzayA9IHRvcmNoLmZyb21fbnVtcHkobWFzaykKCiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgImltYWdlIjogaW1hZ2UsCiAgICAgICAgICAgICJtYXNrIjogbWFzay5sb25nKCksCiAgICAgICAgICAgICJkaXNhc3Rlcl9pZHgiOiB0b3JjaC50ZW5zb3Iocm93WyJkaXNhc3Rlcl9pZHgiXSwgZHR5cGU9dG9yY2gubG9uZyksCiAgICAgICAgICAgICJmaWxlbmFtZSI6IHJvd1siZmlsZW5hbWUiXSwKICAgICAgICB9CgogICAgZGVmIGdldF9zYW1wbGVfd2VpZ2h0cyhzZWxmKToKICAgICAgICAjIFJldHVybiBwZXItc2FtcGxlIHdlaWdodHMgZnJvbSB0aGUgQ1NWIGlmIGF2YWlsYWJsZS4KICAgICAgICAjIFVzZWQgdG8gY29uZmlndXJlIFdlaWdodGVkUmFuZG9tU2FtcGxlci4KICAgICAgICBpZiAic2FtcGxlX3dlaWdodCIgaW4gc2VsZi5kZi5jb2x1bW5zOgogICAgICAgICAgICByZXR1cm4gc2VsZi5kZlsic2FtcGxlX3dlaWdodCJdLnZhbHVlcy5hc3R5cGUobnAuZmxvYXQzMikKICAgICAgICBlbHNlOgogICAgICAgICAgICAjIEZhbGxiYWNrOiB3ZWlnaHQgaGFzYXJsaSBpbWFnZXMgaGlnaGVyCiAgICAgICAgICAgIHdlaWdodHMgPSBucC5vbmVzKGxlbihzZWxmLmRmKSwgZHR5cGU9bnAuZmxvYXQzMikKICAgICAgICAgICAgaWYgImRhbWFnZV9wcmVzZW50IiBpbiBzZWxmLmRmLmNvbHVtbnM6CiAgICAgICAgICAgICAgICB3ZWlnaHRzW3NlbGYuZGZbImRhbWFnZV9wcmVzZW50Il0udmFsdWVzLmFzdHlwZShib29sKV0gPSA1LjAKICAgICAgICAgICAgZWxpZiAiaGFzX2FueV9kYW1hZ2UiIGluIHNlbGYuZGYuY29sdW1uczoKICAgICAgICAgICAgICAgIHdlaWdodHNbc2VsZi5kZlsiaGFzX2FueV9kYW1hZ2UiXS52YWx1ZXMuYXN0eXBlKGJvb2wpXSA9IDUuMAogICAgICAgICAgICByZXR1cm4gd2VpZ2h0cwo=").decode("utf-8")
AUG_V2 = base64.b64decode("IiIiCkFGRVRTT05BUiBhdWdtZW50YXRpb25zIHYyIOKAlCBjbGFzcy1wcmVzZXJ2aW5nLCBsaWdodGVyLgoKQ2hhbmdlcyBmcm9tIHYxOgotIFJFTU9WRUQ6IEdhdXNzTm9pc2UsIENvYXJzZURyb3BvdXQgKGRlc3Ryb3kgZGFtYWdlIHNpZ25hbCkKLSBLRVBUOiByb3RhdGUsIGZsaXAsIHNtYWxsIGJyaWdodG5lc3MvY29udHJhc3QKLSBBRERFRDogaHVlIGppdHRlciBmb3IgZGlzYXN0ZXIgdmFyaWV0eQoiIiIKCmltcG9ydCBhbGJ1bWVudGF0aW9ucyBhcyBBCmZyb20gYWxidW1lbnRhdGlvbnMucHl0b3JjaCBpbXBvcnQgVG9UZW5zb3JWMgoKCklNQUdFTkVUX01FQU4gPSBbMC40ODUsIDAuNDU2LCAwLjQwNl0KSU1BR0VORVRfU1REID0gWzAuMjI5LCAwLjIyNCwgMC4yMjVdCgoKZGVmIGdldF90cmFpbl9hdWdtZW50YXRpb25fdjIoaW1hZ2Vfc2l6ZT01MTIsIG1vZGU9InRlYWNoZXIiKToKICAgICMgQ2xhc3MtcHJlc2VydmluZyBhdWdtZW50YXRpb24gZm9yIHRyYWluaW5nLgogICAgIyBUaGVzZSBhdWdtZW50YXRpb25zIHByZXNlcnZlIGRhbWFnZSBjbGFzc2lmaWNhdGlvbiBzaWduYWwgd2hpbGUKICAgICMgYWRkaW5nIGdlb21ldHJpYyBhbmQgbWlsZCBjb2xvciB2YXJpZXR5LgogICAgdHJhbnNmb3JtcyA9IFsKICAgICAgICAjIEVuc3VyZSBjb25zaXN0ZW50IHNpemUgKGRhdGFzZXQgbWF5IGhhdmUgZG9uZSBidWlsZGluZy1hd2FyZSBjcm9wIGFscmVhZHkpCiAgICAgICAgQS5Mb25nZXN0TWF4U2l6ZShtYXhfc2l6ZT1pbWFnZV9zaXplKSwKICAgICAgICBBLlBhZElmTmVlZGVkKG1pbl9oZWlnaHQ9aW1hZ2Vfc2l6ZSwgbWluX3dpZHRoPWltYWdlX3NpemUsIGJvcmRlcl9tb2RlPTApLAogICAgICAgIEEuUmFuZG9tQ3JvcChoZWlnaHQ9aW1hZ2Vfc2l6ZSwgd2lkdGg9aW1hZ2Vfc2l6ZSksCiAgICAgICAgIyBHZW9tZXRyaWMgKGNsYXNzLXByZXNlcnZpbmcpCiAgICAgICAgQS5SYW5kb21Sb3RhdGU5MChwPTAuNSksCiAgICAgICAgQS5Ib3Jpem9udGFsRmxpcChwPTAuNSksCiAgICAgICAgQS5WZXJ0aWNhbEZsaXAocD0wLjMpLAogICAgICAgICMgTWlsZCBjb2xvciBqaXR0ZXIgKGNhcmVmdWwgbm90IHRvIGRlc3Ryb3kgZGFtYWdlIHNpZ25hbCkKICAgICAgICBBLlJhbmRvbUJyaWdodG5lc3NDb250cmFzdCgKICAgICAgICAgICAgYnJpZ2h0bmVzc19saW1pdD0wLjEsCiAgICAgICAgICAgIGNvbnRyYXN0X2xpbWl0PTAuMSwKICAgICAgICAgICAgcD0wLjMsCiAgICAgICAgKSwKICAgICAgICBBLkh1ZVNhdHVyYXRpb25WYWx1ZSgKICAgICAgICAgICAgaHVlX3NoaWZ0X2xpbWl0PTUsCiAgICAgICAgICAgIHNhdF9zaGlmdF9saW1pdD0xMCwKICAgICAgICAgICAgdmFsX3NoaWZ0X2xpbWl0PTUsCiAgICAgICAgICAgIHA9MC4yLAogICAgICAgICksCiAgICAgICAgIyBOb3JtYWxpemUgKyB0byB0ZW5zb3IKICAgICAgICBBLk5vcm1hbGl6ZShtZWFuPUlNQUdFTkVUX01FQU4sIHN0ZD1JTUFHRU5FVF9TVEQpLAogICAgICAgIFRvVGVuc29yVjIoKSwKICAgIF0KCiAgICBpZiBtb2RlID09ICJ0ZWFjaGVyIjoKICAgICAgICByZXR1cm4gQS5Db21wb3NlKHRyYW5zZm9ybXMsIGFkZGl0aW9uYWxfdGFyZ2V0cz17InByZSI6ICJpbWFnZSJ9KQogICAgcmV0dXJuIEEuQ29tcG9zZSh0cmFuc2Zvcm1zKQoKCmRlZiBnZXRfdmFsX2F1Z21lbnRhdGlvbl92MihpbWFnZV9zaXplPTUxMiwgbW9kZT0idGVhY2hlciIpOgogICAgIyBWYWxpZGF0aW9uOiBtaW5pbWFsIOKAlCByZXNpemUgKyBub3JtYWxpemUuCiAgICB0cmFuc2Zvcm1zID0gWwogICAgICAgIEEuTG9uZ2VzdE1heFNpemUobWF4X3NpemU9aW1hZ2Vfc2l6ZSksCiAgICAgICAgQS5QYWRJZk5lZWRlZChtaW5faGVpZ2h0PWltYWdlX3NpemUsIG1pbl93aWR0aD1pbWFnZV9zaXplLCBib3JkZXJfbW9kZT0wKSwKICAgICAgICBBLk5vcm1hbGl6ZShtZWFuPUlNQUdFTkVUX01FQU4sIHN0ZD1JTUFHRU5FVF9TVEQpLAogICAgICAgIFRvVGVuc29yVjIoKSwKICAgIF0KCiAgICBpZiBtb2RlID09ICJ0ZWFjaGVyIjoKICAgICAgICByZXR1cm4gQS5Db21wb3NlKHRyYW5zZm9ybXMsIGFkZGl0aW9uYWxfdGFyZ2V0cz17InByZSI6ICJpbWFnZSJ9KQogICAgcmV0dXJuIEEuQ29tcG9zZSh0cmFuc2Zvcm1zKQo=").decode("utf-8")
METRICS_PY = base64.b64decode("IiIiCkFGRVRTT05BUiBNZXRyaWNzIOKAlCBzZWdtZW50YXRpb24gKG1Jb1UsIEYxKSBhbmQgY2xhc3NpZmljYXRpb24gKGFjY3VyYWN5KS4KIiIiCgppbXBvcnQgdG9yY2gKaW1wb3J0IG51bXB5IGFzIG5wCgoKY2xhc3MgU2VnbWVudGF0aW9uTWV0cmljczoKICAgICIiIgogICAgU3RyZWFtaW5nIGNvbmZ1c2lvbiBtYXRyaXggZm9yIG11bHRpLWNsYXNzIHNlZ21lbnRhdGlvbi4KCiAgICBVc2FnZToKICAgICAgICBtZXRyaWNzID0gU2VnbWVudGF0aW9uTWV0cmljcyhudW1fY2xhc3Nlcz01KQogICAgICAgIGZvciBiYXRjaCBpbiBsb2FkZXI6CiAgICAgICAgICAgIHByZWRzID0gbW9kZWwoYmF0Y2gpLmFyZ21heChkaW09MSkKICAgICAgICAgICAgbWV0cmljcy51cGRhdGUocHJlZHMsIGJhdGNoWydtYXNrJ10pCiAgICAgICAgc2NvcmVzID0gbWV0cmljcy5jb21wdXRlKCkKICAgICAgICAjIHNjb3JlczogeydtaW91JzogZmxvYXQsICdpb3VfcGVyX2NsYXNzJzogWy4uLl0sICdhY2N1cmFjeSc6IGZsb2F0fQogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIG51bV9jbGFzc2VzLCBpZ25vcmVfaW5kZXg9Tm9uZSk6CiAgICAgICAgc2VsZi5udW1fY2xhc3NlcyA9IG51bV9jbGFzc2VzCiAgICAgICAgc2VsZi5pZ25vcmVfaW5kZXggPSBpZ25vcmVfaW5kZXgKICAgICAgICBzZWxmLnJlc2V0KCkKCiAgICBkZWYgcmVzZXQoc2VsZik6CiAgICAgICAgc2VsZi5jb25mdXNpb24gPSBucC56ZXJvcygoc2VsZi5udW1fY2xhc3Nlcywgc2VsZi5udW1fY2xhc3NlcyksIGR0eXBlPW5wLmludDY0KQoKICAgIEB0b3JjaC5ub19ncmFkKCkKICAgIGRlZiB1cGRhdGUoc2VsZiwgcHJlZHMsIHRhcmdldHMpOgogICAgICAgICIiIgogICAgICAgIHByZWRzOiAgIFtCLCBILCBXXSBsb25nIHRlbnNvcgogICAgICAgIHRhcmdldHM6IFtCLCBILCBXXSBsb25nIHRlbnNvcgogICAgICAgICIiIgogICAgICAgIGlmIGlzaW5zdGFuY2UocHJlZHMsIHRvcmNoLlRlbnNvcik6CiAgICAgICAgICAgIHByZWRzID0gcHJlZHMuZGV0YWNoKCkuY3B1KCkubnVtcHkoKQogICAgICAgIGlmIGlzaW5zdGFuY2UodGFyZ2V0cywgdG9yY2guVGVuc29yKToKICAgICAgICAgICAgdGFyZ2V0cyA9IHRhcmdldHMuZGV0YWNoKCkuY3B1KCkubnVtcHkoKQoKICAgICAgICBwcmVkcyA9IHByZWRzLmZsYXR0ZW4oKQogICAgICAgIHRhcmdldHMgPSB0YXJnZXRzLmZsYXR0ZW4oKQoKICAgICAgICBpZiBzZWxmLmlnbm9yZV9pbmRleCBpcyBub3QgTm9uZToKICAgICAgICAgICAgbWFzayA9IHRhcmdldHMgIT0gc2VsZi5pZ25vcmVfaW5kZXgKICAgICAgICAgICAgcHJlZHMgPSBwcmVkc1ttYXNrXQogICAgICAgICAgICB0YXJnZXRzID0gdGFyZ2V0c1ttYXNrXQoKICAgICAgICAjIEZpbHRlciBvdXQtb2YtcmFuZ2UgcHJlZGljdGlvbnMKICAgICAgICB2YWxpZCA9ICh0YXJnZXRzID49IDApICYgKHRhcmdldHMgPCBzZWxmLm51bV9jbGFzc2VzKQogICAgICAgIHByZWRzID0gcHJlZHNbdmFsaWRdCiAgICAgICAgdGFyZ2V0cyA9IHRhcmdldHNbdmFsaWRdCgogICAgICAgICMgQnVpbGQgY29uZnVzaW9uIG1hdHJpeCB1c2luZyBiaW5jb3VudCB0cmljayAoZmFzdCkKICAgICAgICBpZHggPSBzZWxmLm51bV9jbGFzc2VzICogdGFyZ2V0cyArIHByZWRzCiAgICAgICAgYmluYyA9IG5wLmJpbmNvdW50KGlkeCwgbWlubGVuZ3RoPXNlbGYubnVtX2NsYXNzZXMgKiogMikKICAgICAgICBzZWxmLmNvbmZ1c2lvbiArPSBiaW5jLnJlc2hhcGUoc2VsZi5udW1fY2xhc3Nlcywgc2VsZi5udW1fY2xhc3NlcykKCiAgICBkZWYgY29tcHV0ZShzZWxmKToKICAgICAgICAiIiIKICAgICAgICBSZXR1cm5zOgogICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAnbWlvdSc6ICAgICAgICAgICBmbG9hdCwKICAgICAgICAgICAgICAgICdtaW91X25vX2JnJzogICAgIGZsb2F0LCAgIyBtSW9VIHdpdGhvdXQgYmFja2dyb3VuZCBjbGFzcwogICAgICAgICAgICAgICAgJ2lvdV9wZXJfY2xhc3MnOiAgbGlzdCBvZiBmbG9hdHMsCiAgICAgICAgICAgICAgICAnYWNjdXJhY3knOiAgICAgICBmbG9hdCwKICAgICAgICAgICAgICAgICdmMV9wZXJfY2xhc3MnOiAgIGxpc3Qgb2YgZmxvYXRzLAogICAgICAgICAgICAgICAgJ21mMSc6ICAgICAgICAgICAgZmxvYXQsCiAgICAgICAgICAgIH0KICAgICAgICAiIiIKICAgICAgICBjbSA9IHNlbGYuY29uZnVzaW9uLmFzdHlwZShucC5mbG9hdDY0KQoKICAgICAgICAjIElvVSBwZXIgY2xhc3M6IFRQIC8gKFRQICsgRlAgKyBGTikKICAgICAgICB0cCA9IG5wLmRpYWcoY20pCiAgICAgICAgZnAgPSBjbS5zdW0oYXhpcz0wKSAtIHRwCiAgICAgICAgZm4gPSBjbS5zdW0oYXhpcz0xKSAtIHRwCgogICAgICAgIGlvdSA9IHRwIC8gbnAubWF4aW11bSh0cCArIGZwICsgZm4sIDEuMCkKICAgICAgICBpb3VbKHRwICsgZnAgKyBmbikgPT0gMF0gPSBmbG9hdCgibmFuIikgICMgY2xhc3MgbmV2ZXIgc2VlbgoKICAgICAgICAjIEYxIHBlciBjbGFzcwogICAgICAgIHByZWNpc2lvbiA9IHRwIC8gbnAubWF4aW11bSh0cCArIGZwLCAxLjApCiAgICAgICAgcmVjYWxsID0gdHAgLyBucC5tYXhpbXVtKHRwICsgZm4sIDEuMCkKICAgICAgICBmMSA9IDIgKiBwcmVjaXNpb24gKiByZWNhbGwgLyBucC5tYXhpbXVtKHByZWNpc2lvbiArIHJlY2FsbCwgMWUtOCkKICAgICAgICBmMVsodHAgKyBmcCArIGZuKSA9PSAwXSA9IGZsb2F0KCJuYW4iKQoKICAgICAgICAjIFBpeGVsIGFjY3VyYWN5CiAgICAgICAgYWNjdXJhY3kgPSB0cC5zdW0oKSAvIG1heChjbS5zdW0oKSwgMS4wKQoKICAgICAgICBtaW91ID0gbnAubmFubWVhbihpb3UpCiAgICAgICAgbWlvdV9ub19iZyA9IG5wLm5hbm1lYW4oaW91WzE6XSkgICMgc2tpcCBjbGFzcyAwIChiYWNrZ3JvdW5kKQogICAgICAgIG1mMSA9IG5wLm5hbm1lYW4oZjEpCgogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICJtaW91IjogZmxvYXQobWlvdSksCiAgICAgICAgICAgICJtaW91X25vX2JnIjogZmxvYXQobWlvdV9ub19iZyksCiAgICAgICAgICAgICJpb3VfcGVyX2NsYXNzIjogaW91LnRvbGlzdCgpLAogICAgICAgICAgICAiYWNjdXJhY3kiOiBmbG9hdChhY2N1cmFjeSksCiAgICAgICAgICAgICJmMV9wZXJfY2xhc3MiOiBmMS50b2xpc3QoKSwKICAgICAgICAgICAgIm1mMSI6IGZsb2F0KG1mMSksCiAgICAgICAgfQoKCmNsYXNzIENsYXNzaWZpY2F0aW9uTWV0cmljczoKICAgICIiIgogICAgU2ltcGxlIGFjY3VyYWN5ICsgcGVyLWNsYXNzIGFjY3VyYWN5IGZvciBpbWFnZS1sZXZlbCBjbGFzc2lmaWNhdGlvbi4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBudW1fY2xhc3Nlcyk6CiAgICAgICAgc2VsZi5udW1fY2xhc3NlcyA9IG51bV9jbGFzc2VzCiAgICAgICAgc2VsZi5yZXNldCgpCgogICAgZGVmIHJlc2V0KHNlbGYpOgogICAgICAgIHNlbGYuY29ycmVjdCA9IDAKICAgICAgICBzZWxmLnRvdGFsID0gMAogICAgICAgIHNlbGYucGVyX2NsYXNzX2NvcnJlY3QgPSBucC56ZXJvcyhzZWxmLm51bV9jbGFzc2VzLCBkdHlwZT1ucC5pbnQ2NCkKICAgICAgICBzZWxmLnBlcl9jbGFzc190b3RhbCA9IG5wLnplcm9zKHNlbGYubnVtX2NsYXNzZXMsIGR0eXBlPW5wLmludDY0KQoKICAgIEB0b3JjaC5ub19ncmFkKCkKICAgIGRlZiB1cGRhdGUoc2VsZiwgcHJlZHMsIHRhcmdldHMpOgogICAgICAgICIiIgogICAgICAgIHByZWRzOiAgIFtCXSBsb25nIHRlbnNvciAoYXJnbWF4IGFscmVhZHkgZG9uZSkKICAgICAgICB0YXJnZXRzOiBbQl0gbG9uZyB0ZW5zb3IKICAgICAgICAiIiIKICAgICAgICBpZiBpc2luc3RhbmNlKHByZWRzLCB0b3JjaC5UZW5zb3IpOgogICAgICAgICAgICBwcmVkcyA9IHByZWRzLmRldGFjaCgpLmNwdSgpLm51bXB5KCkKICAgICAgICBpZiBpc2luc3RhbmNlKHRhcmdldHMsIHRvcmNoLlRlbnNvcik6CiAgICAgICAgICAgIHRhcmdldHMgPSB0YXJnZXRzLmRldGFjaCgpLmNwdSgpLm51bXB5KCkKCiAgICAgICAgc2VsZi5jb3JyZWN0ICs9IChwcmVkcyA9PSB0YXJnZXRzKS5zdW0oKQogICAgICAgIHNlbGYudG90YWwgKz0gbGVuKHRhcmdldHMpCgogICAgICAgIGZvciBjIGluIHJhbmdlKHNlbGYubnVtX2NsYXNzZXMpOgogICAgICAgICAgICBtYXNrID0gdGFyZ2V0cyA9PSBjCiAgICAgICAgICAgIHNlbGYucGVyX2NsYXNzX2NvcnJlY3RbY10gKz0gKHByZWRzW21hc2tdID09IGMpLnN1bSgpCiAgICAgICAgICAgIHNlbGYucGVyX2NsYXNzX3RvdGFsW2NdICs9IG1hc2suc3VtKCkKCiAgICBkZWYgY29tcHV0ZShzZWxmKToKICAgICAgICBhY2N1cmFjeSA9IHNlbGYuY29ycmVjdCAvIG1heChzZWxmLnRvdGFsLCAxKQogICAgICAgIHBlcl9jbGFzc19hY2MgPSBzZWxmLnBlcl9jbGFzc19jb3JyZWN0IC8gbnAubWF4aW11bShzZWxmLnBlcl9jbGFzc190b3RhbCwgMSkKICAgICAgICByZXR1cm4gewogICAgICAgICAgICAiYWNjdXJhY3kiOiBmbG9hdChhY2N1cmFjeSksCiAgICAgICAgICAgICJwZXJfY2xhc3NfYWNjdXJhY3kiOiBwZXJfY2xhc3NfYWNjLnRvbGlzdCgpLAogICAgICAgICAgICAiYmFsYW5jZWRfYWNjdXJhY3kiOiBmbG9hdChucC5uYW5tZWFuKHBlcl9jbGFzc19hY2MpKSwKICAgICAgICB9Cg==").decode("utf-8")

files_to_write = {
    "models_v2.py": MODELS_V2,
    "losses_v2.py": LOSSES_V2,
    "dataset_v2.py": DATASET_V2,
    "augmentations_v2.py": AUG_V2,
    "metrics.py": METRICS_PY,  # aynı kalıyor, yeniden yazıyoruz emin olmak için
}

print("📝 v2 modülleri yazılıyor...")
print("=" * 50)
for filename, content in files_to_write.items():
    full_path = os.path.join(SRC_DIR, filename)
    with open(full_path, "w") as f:
        f.write(content)
    size_kb = os.path.getsize(full_path) / 1024
    print(f"  ✅ {filename}  ({size_kb:.1f} KB)")

import sys
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("\n🧪 Import testi...")
import importlib
import models_v2, losses_v2, dataset_v2, augmentations_v2, metrics
importlib.reload(models_v2)
importlib.reload(losses_v2)
importlib.reload(dataset_v2)
importlib.reload(augmentations_v2)
importlib.reload(metrics)

from models_v2 import SiameseTeacherSegformer
from losses_v2 import TeacherLossV2, ComboDamageLoss, derive_change_mask_v2
from dataset_v2 import XBDDatasetV2
from augmentations_v2 import get_train_augmentation_v2, get_val_augmentation_v2
from metrics import SegmentationMetrics, ClassificationMetrics
print("  ✅ Tüm importlar başarılı")

📝 v2 modülleri yazılıyor...
  ✅ models_v2.py  (6.6 KB)
  ✅ losses_v2.py  (8.8 KB)
  ✅ dataset_v2.py  (6.2 KB)
  ✅ augmentations_v2.py  (2.1 KB)
  ✅ metrics.py  (4.5 KB)

🧪 Import testi...
  ✅ Tüm importlar başarılı


## 3️⃣ Siamese Öğretmen Modelini İnşa Et

### Mimari Özeti

```
Pre [B,3,H,W]  ─┐
                ├─ Shared Encoder (SegFormer-B3) ─┬─ features_pre (4 stage)
Post [B,3,H,W] ─┘                                 ├─ features_post (4 stage)
                                                   │
                                                   ▼
                                              Fusion Modules
                                     (concat + diff → 1x1 conv)
                                                   │
                                                   ▼
                                         fused features (4 stage)
                                                   │
                        ┌──────────────────────────┼──────────────────────────┐
                        ▼                          ▼                          ▼
                   Damage Head              Change Head                 Disaster Head
                   (SegFormer decoder)      (Conv decoder)             (GAP + Linear)
                        │                          │                          │
                        ▼                          ▼                          ▼
                   [B,6,H,W]                  [B,2,H,W]                    [B,5]
```

### Neden Shared Encoder?

Eski modelde 6 kanalı tek bir conv'a yediriyorduk. Model "pre" ve "post"un hangileri olduğunu bile ayırt edemiyordu. Şimdi:

1. Pre ve post **ayrı forward pass** ile encoder'dan geçiyor → iki ayrı feature seti
2. Feature'lar her aşamada **fuse ediliyor** (pre, post, |diff| concat + 1x1 conv)
3. Encoder ağırlıkları **paylaşılıyor** — bu true Siamese (pre ve post için aynı "anlama" fonksiyonu)

### SegFormer-B3 vs B5

| Özellik | B5 | B3 |
|---|---|---|
| Parametre | ~85M | ~45M |
| Hız | 1x | ~2x |
| VRAM (batch 32) | ~50 GB | ~25 GB |

B3 bizim veri boyutumuz için ideal — overfit olmadan yeterli kapasite.

In [3]:
import sys
sys.path.insert(0, SRC_DIR)

import importlib
import models_v2
importlib.reload(models_v2)
from models_v2 import SiameseTeacherSegformer

print("🏗️  Siamese SegFormer-B3 inşa ediliyor...")
print("=" * 50)
print("   İlk seferinde ~170 MB indirilecek...")

model = SiameseTeacherSegformer(
    backbone_name="nvidia/segformer-b3-finetuned-ade-512-512",
    num_damage_classes=6,  # 0=bg + 5 damage (incl un-classified)
    num_disaster_classes=5,
    pretrained=True,
)

# Gradient checkpointing (belleği optimize et)
if USE_GRADIENT_CHECKPOINTING:
    enabled = model.enable_gradient_checkpointing()
    if enabled:
        print("   ✅ Gradient checkpointing aktif (belleği ~%30 azaltır)")
    else:
        print("   ⚠️  Gradient checkpointing etkinleştirilemedi")

model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✅ Model GPU'ya taşındı")
print(f"   Toplam parametre: {total_params:,} (~{total_params/1e6:.1f}M)")

# Forward test
print("\n🧪 Forward pass testi...")
model.eval()
amp_dtype = torch.bfloat16 if PRECISION == "bf16" else torch.float16

with torch.no_grad():
    dummy = torch.randn(2, 6, 512, 512, device=device)
    with torch.autocast(device_type="cuda", dtype=amp_dtype):
        out = model(dummy)

print(f"   damage_logits:   {tuple(out['damage_logits'].shape)}  (expected: (2, 6, 512, 512))")
print(f"   change_logits:   {tuple(out['change_logits'].shape)}  (expected: (2, 2, 512, 512))")
print(f"   disaster_logits: {tuple(out['disaster_logits'].shape)}  (expected: (2, 5))")

assert out["damage_logits"].shape == (2, 6, 512, 512)
assert out["change_logits"].shape == (2, 2, 512, 512)
assert out["disaster_logits"].shape == (2, 5)
print("\n✅ Tüm shape'ler doğru")

mem_gb = torch.cuda.memory_allocated() / 1024**3
print(f"\n💾 GPU bellek: {mem_gb:.2f} GB / {vram_gb} GB")

🏗️  Siamese SegFormer-B3 inşa ediliyor...
   İlk seferinde ~170 MB indirilecek...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/190M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/644 [00:00<?, ?it/s]

SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b3-finetuned-ade-512-512
Key                           | Status   |                                                                                                   
------------------------------+----------+---------------------------------------------------------------------------------------------------
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([150]) vs model:torch.Size([6])                      
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([150, 768, 1, 1]) vs model:torch.Size([6, 768, 1, 1])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


model.safetensors:   0%|          | 0.00/189M [00:00<?, ?B/s]


✅ Model GPU'ya taşındı
   Toplam parametre: 49,697,741 (~49.7M)

🧪 Forward pass testi...
   damage_logits:   (2, 6, 512, 512)  (expected: (2, 6, 512, 512))
   change_logits:   (2, 2, 512, 512)  (expected: (2, 2, 512, 512))
   disaster_logits: (2, 5)  (expected: (2, 5))

✅ Tüm shape'ler doğru

💾 GPU bellek: 0.25 GB / 79.2 GB


## 4️⃣ DataLoader'lar (WeightedRandomSampler ile)

### Kritik değişiklik: WeightedRandomSampler

Notebook 1b'de her train örneği için bir `sample_weight` hesaplamıştık. Hasarlı görüntüler 6-15x, hasarsızlar 1x. Bu sayede model her epoch'ta **hasarlı örnekleri çok daha sık** görür.

### Building-aware crop

Eski `XBDDataset` RandomCrop kullanıyordu → cropların çoğu boş tarla oluyordu. Yeni `XBDDatasetV2` %80 ihtimalle **bina merkezli** crop yapıyor. Bu, model her batch'te **dolu** örnekler görmesini sağlıyor.

In [4]:
import sys
sys.path.insert(0, SRC_DIR)

import importlib
import dataset_v2, augmentations_v2, losses_v2
importlib.reload(dataset_v2)
importlib.reload(augmentations_v2)
importlib.reload(losses_v2)

from dataset_v2 import XBDDatasetV2
from augmentations_v2 import get_train_augmentation_v2, get_val_augmentation_v2
from losses_v2 import derive_change_mask_v2
from torch.utils.data import DataLoader, WeightedRandomSampler

print("📦 DataLoader'lar kuruluyor...")
print("=" * 50)

# === Datasets ===
train_aug = get_train_augmentation_v2(image_size=512, mode="teacher")
val_aug = get_val_augmentation_v2(image_size=512, mode="teacher")

train_ds = XBDDatasetV2(
    csv_path=os.path.join(DATA_SPLITS, "train_v2.csv"),
    mode="teacher",
    augmentation=train_aug,
    building_aware_crop=True,
    building_crop_prob=0.8,
)

val_ds = XBDDatasetV2(
    csv_path=os.path.join(DATA_SPLITS, "val_v2.csv"),
    mode="teacher",
    augmentation=val_aug,
    building_aware_crop=False,  # val'de crop yok, full image
)

print(f"   train: {len(train_ds)} örnek")
print(f"   val:   {len(val_ds)} örnek")

# === WeightedRandomSampler for training ===
train_sample_weights = train_ds.get_sample_weights()
print(f"\n   Sample weights:")
print(f"     min={train_sample_weights.min():.2f}")
print(f"     max={train_sample_weights.max():.2f}")
print(f"     mean={train_sample_weights.mean():.2f}")

# num_samples: kaç sample çekilecek her epoch'ta
# Genelde len(dataset) kadar yapıyoruz, ama hasarlı örnekleri daha sık çekmek için
# biraz daha fazla çekebiliriz (2x)
num_samples_per_epoch = len(train_ds)

train_sampler = WeightedRandomSampler(
    weights=train_sample_weights.tolist(),
    num_samples=num_samples_per_epoch,
    replacement=True,
)
print(f"\n   WeightedRandomSampler ile {num_samples_per_epoch} örnek / epoch")

# === Collate function: change mask türetme ===
def teacher_collate_fn(batch):
    images = torch.stack([item["image"] for item in batch])
    damage_masks = torch.stack([item["mask"] for item in batch])
    disaster_idxs = torch.stack([item["disaster_idx"] for item in batch])
    change_masks = derive_change_mask_v2(damage_masks)

    return {
        "image": images,
        "damage_mask": damage_masks,
        "change_mask": change_masks,
        "disaster_idx": disaster_idxs,
    }

# === DataLoaders ===
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,  # Shuffle YOK, sampler kullanıyoruz
    num_workers=NUM_WORKERS,
    collate_fn=teacher_collate_fn,
    pin_memory=True,
    drop_last=True,
    persistent_workers=True if NUM_WORKERS > 0 else False,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=teacher_collate_fn,
    pin_memory=True,
    persistent_workers=True if NUM_WORKERS > 0 else False,
)

print(f"\n   train batch sayısı: {len(train_loader)}")
print(f"   val batch sayısı:   {len(val_loader)}")

# === Sample batch test ===
print("\n🧪 Sample batch testi...")
batch = next(iter(train_loader))
print(f"   image:        {tuple(batch['image'].shape)}")
print(f"   damage_mask:  {tuple(batch['damage_mask'].shape)}")
print(f"   change_mask:  {tuple(batch['change_mask'].shape)}")
print(f"   disaster_idx: {tuple(batch['disaster_idx'].shape)}")
print(f"   damage unique: {sorted(batch['damage_mask'].unique().tolist())}")
print(f"   change unique: {sorted(batch['change_mask'].unique().tolist())}")
print("\n✅ DataLoader hazır")

📦 DataLoader'lar kuruluyor...
   train: 1959 örnek
   val:   420 örnek

   Sample weights:
     min=1.00
     max=5.00
     mean=3.15

   WeightedRandomSampler ile 1959 örnek / epoch

   train batch sayısı: 122
   val batch sayısı:   27

🧪 Sample batch testi...
   image:        (16, 6, 512, 512)
   damage_mask:  (16, 512, 512)
   change_mask:  (16, 512, 512)
   disaster_idx: (16,)
   damage unique: [0, 1, 2, 3, 4, 5]
   change unique: [0, 1]

✅ DataLoader hazır


## 5️⃣ Combo Loss + Optimizer + Scheduler

### Loss

```
L_total = 0.70·L_damage_combo + 0.20·L_ce_change + 0.10·L_ce_disaster

L_damage_combo = 0.5·Focal + 0.3·Dice + 0.2·Boundary
```

**Class weights** (Notebook 1b'den hesaplanan, minor biraz daha agresif):

```python
damage_class_weights = [0.05, 1.0, 10.0, 7.0, 10.0, 0.5]
                       # bg    no    minor  major  dest  uncls
```

- `bg=0.05` — arka plan aşırı yaygın, düşük weight
- `no_damage=1.0` — baseline
- `minor=10.0` — en nadir sınıf, agresif boost (Notebook 1b: 8.5 → 10.0)
- `major=7.0` — (Notebook 1b: 6.5 → 7.0)
- `destroyed=10.0` — destroyed zor öğrenilen sınıf
- `un-classified=0.5` — az önemli, model buna kaymasın

### Optimizer

AdamW, lr=**1e-4** (eskisinden biraz yüksek, 6e-5 → 1e-4), weight_decay=0.01

### Scheduler

Cosine annealing with **warmup**: ilk 3 epoch linear warmup → sonra cosine.

In [5]:
import sys
sys.path.insert(0, SRC_DIR)

import importlib
import losses_v2, metrics
importlib.reload(losses_v2)
importlib.reload(metrics)

from losses_v2 import TeacherLossV2
from metrics import SegmentationMetrics, ClassificationMetrics
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR, CosineAnnealingLR
import math

# === Hyperparameters ===
LR = 1e-4
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 80
WARMUP_EPOCHS = 3

# === TEST_MODE ===
TEST_MODE = False  # True → 3 epoch + 15 batch/epoch

if TEST_MODE:
    NUM_EPOCHS = 3
    print("⚠️  TEST_MODE ON: 3 epoch, limited batches")

print(f"\n⚙️  Hyperparameters:")
print(f"  learning_rate  = {LR}")
print(f"  weight_decay   = {WEIGHT_DECAY}")
print(f"  epochs         = {NUM_EPOCHS}")
print(f"  warmup_epochs  = {WARMUP_EPOCHS}")
print(f"  batch_size     = {BATCH_SIZE}")

# === Class weights (Notebook 1b analizine göre) ===
damage_class_weights = [0.05, 1.0, 10.0, 7.0, 10.0, 0.5]

print(f"\n📐 Damage class weights:")
class_names = ["bg", "no_damage", "minor", "major", "destroyed", "un-classified"]
for name, w in zip(class_names, damage_class_weights):
    print(f"  {name:<15s} {w:.2f}")

# === Loss ===
criterion = TeacherLossV2(
    num_damage_classes=6,
    damage_weight=0.70,
    change_weight=0.20,
    disaster_weight=0.10,
    focal_gamma=2.0,
    damage_class_weights=damage_class_weights,
).to(device)

print(f"\n📉 Loss: TeacherLossV2")
print(f"  Damage combo: 0.5·Focal + 0.3·Dice + 0.2·Boundary")
print(f"  Task weights: 0.70·damage + 0.20·change + 0.10·disaster")

# === Optimizer ===
optimizer = AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999),
)
print(f"\n⚙️  Optimizer: AdamW")

# === Warmup + Cosine Scheduler ===
# NOT: Gradient accumulation kullandığımız için scheduler adımları
# "kaç optimizer.step() çağrılır" üzerinden hesaplanır
steps_per_epoch = len(train_loader) // GRAD_ACCUM_STEPS
total_steps = NUM_EPOCHS * steps_per_epoch
warmup_steps = WARMUP_EPOCHS * steps_per_epoch

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    else:
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)
print(f"📅 Scheduler: Warmup ({WARMUP_EPOCHS} epochs) → Cosine")
print(f"   steps/epoch = {steps_per_epoch} (after {GRAD_ACCUM_STEPS}x accumulation)")
print(f"   total_steps = {total_steps}")

# === Metrics ===
train_seg_metrics = SegmentationMetrics(num_classes=6)
val_seg_metrics = SegmentationMetrics(num_classes=6)
val_disaster_metrics = ClassificationMetrics(num_classes=5)

print("\n✅ Eğitim altyapısı hazır")


⚙️  Hyperparameters:
  learning_rate  = 0.0001
  weight_decay   = 0.01
  epochs         = 80
  warmup_epochs  = 3
  batch_size     = 16

📐 Damage class weights:
  bg              0.05
  no_damage       1.00
  minor           10.00
  major           7.00
  destroyed       10.00
  un-classified   0.50

📉 Loss: TeacherLossV2
  Damage combo: 0.5·Focal + 0.3·Dice + 0.2·Boundary
  Task weights: 0.70·damage + 0.20·change + 0.10·disaster

⚙️  Optimizer: AdamW
📅 Scheduler: Warmup (3 epochs) → Cosine
   steps/epoch = 40 (after 3x accumulation)
   total_steps = 3200

✅ Eğitim altyapısı hazır


## 6️⃣ Eğitim Döngüsü

### Aynı yapı, daha fazla log

Eğitim döngüsü Notebook 2'dekiyle benzer ama:

1. **Combo loss komponentlerini ayrı logluyor** (focal, dice, boundary)
2. **Her 5 epoch'ta val prediction görseli** kaydediyor (eğitim sırasında ilerlemeyi görmek için)
3. **Early stopping** yok — plan'a göre 80 epoch'u tam çalıştır
4. **Best model'i mIoU_no_bg'ye göre seçiyor** (bg hariç mIoU)

### Takip edilecek metrikler

- `train_loss_total` — toplam loss
- `train_loss_focal`, `train_loss_dice`, `train_loss_boundary` — combo componentleri
- `val_loss`, `val_mIoU`, `val_mIoU_no_bg`
- `val_f1_per_class` — her sınıfın F1 skoru ayrı ayrı

### Beklenen süre

H100'de ~2-3 saat (B3 hızlı).

In [ ]:
import time
import json
from tqdm.auto import tqdm

# Checkpoint klasörü
os.makedirs(CKPT_TEACHER, exist_ok=True)

# Metrik geçmişi
history = {
    "train_loss": [], "train_focal": [], "train_dice": [], "train_boundary": [],
    "train_change": [], "train_disaster": [],
    "val_loss": [], "val_miou": [], "val_miou_no_bg": [], "val_mf1": [],
    "val_disaster_acc": [], "val_iou_per_class": [], "val_f1_per_class": [],
    "lr": [], "epoch_time": [],
}

best_miou_no_bg = 0.0
print(f"\n🚀 EĞİTİM BAŞLIYOR (v2)")
print(f"   Toplam epoch: {NUM_EPOCHS}")
print(f"   Train batch:  {len(train_loader)}")
print(f"   Val batch:    {len(val_loader)}")
print(f"   Checkpoint:   {CKPT_TEACHER}")
print("=" * 75)

amp_dtype = torch.bfloat16 if PRECISION == "bf16" else torch.float16

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()

    # ========== TRAIN ==========
    model.train()
    train_seg_metrics.reset()

    train_losses = {"total": 0.0, "focal": 0.0, "dice": 0.0, "boundary": 0.0,
                    "change": 0.0, "disaster": 0.0}
    n_batches = 0
    optim_step_count = 0

    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [train]", leave=False)

    # Gradient accumulation: her GRAD_ACCUM_STEPS'te bir optimizer.step()
    optimizer.zero_grad(set_to_none=True)

    for batch_idx, batch in enumerate(train_pbar):
        if TEST_MODE and batch_idx >= 15:
            break

        images = batch["image"].to(device, non_blocking=True)
        damage_mask = batch["damage_mask"].to(device, non_blocking=True)
        change_mask = batch["change_mask"].to(device, non_blocking=True)
        disaster_idx = batch["disaster_idx"].to(device, non_blocking=True)

        targets = {
            "damage_mask": damage_mask,
            "change_mask": change_mask,
            "disaster_idx": disaster_idx,
        }

        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            outputs = model(images)
            loss_dict = criterion(outputs, targets)
            # Normalize loss by accumulation steps (so gradients sum correctly)
            loss = loss_dict["total"] / GRAD_ACCUM_STEPS

        loss.backward()

        # Optimizer step only every GRAD_ACCUM_STEPS batches
        if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            optim_step_count += 1

        # Loss komponentleri (unscaled değerlerle)
        train_losses["total"] += loss_dict["total"].item()
        train_losses["focal"] += loss_dict["damage_focal"].item()
        train_losses["dice"] += loss_dict["damage_dice"].item()
        train_losses["boundary"] += loss_dict["damage_boundary"].item()
        train_losses["change"] += loss_dict["change"].item()
        train_losses["disaster"] += loss_dict["disaster"].item()
        n_batches += 1

        # Train metrics (ilk 30 batch yeterli hız için)
        if batch_idx < 30:
            with torch.no_grad():
                preds = outputs["damage_logits"].argmax(dim=1)
                train_seg_metrics.update(preds, damage_mask)

        train_pbar.set_postfix({"loss": f"{loss_dict['total'].item():.3f}"})

    # Epoch sonunda kalan gradient'ları commit et (eğer accumulation tam bitmezse)
    if (n_batches % GRAD_ACCUM_STEPS) != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

    for k in train_losses:
        train_losses[k] /= max(n_batches, 1)

    # ========== VAL ==========
    model.eval()
    val_seg_metrics.reset()
    val_disaster_metrics.reset()
    val_loss_total = 0.0
    n_val = 0

    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [val]  ", leave=False)
    with torch.no_grad():
        for batch_idx, batch in enumerate(val_pbar):
            if TEST_MODE and batch_idx >= 10:
                break

            images = batch["image"].to(device, non_blocking=True)
            damage_mask = batch["damage_mask"].to(device, non_blocking=True)
            change_mask = batch["change_mask"].to(device, non_blocking=True)
            disaster_idx = batch["disaster_idx"].to(device, non_blocking=True)

            targets = {
                "damage_mask": damage_mask,
                "change_mask": change_mask,
                "disaster_idx": disaster_idx,
            }

            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                outputs = model(images)
                loss_dict = criterion(outputs, targets)

            val_loss_total += loss_dict["total"].item()
            n_val += 1

            preds = outputs["damage_logits"].argmax(dim=1)
            val_seg_metrics.update(preds, damage_mask)

            disaster_preds = outputs["disaster_logits"].argmax(dim=1)
            val_disaster_metrics.update(disaster_preds, disaster_idx)

    val_loss_avg = val_loss_total / max(n_val, 1)
    val_scores = val_seg_metrics.compute()
    val_disaster_scores = val_disaster_metrics.compute()
    epoch_time = (time.time() - epoch_start) / 60

    # Log
    history["train_loss"].append(train_losses["total"])
    history["train_focal"].append(train_losses["focal"])
    history["train_dice"].append(train_losses["dice"])
    history["train_boundary"].append(train_losses["boundary"])
    history["train_change"].append(train_losses["change"])
    history["train_disaster"].append(train_losses["disaster"])
    history["val_loss"].append(val_loss_avg)
    history["val_miou"].append(val_scores["miou"])
    history["val_miou_no_bg"].append(val_scores["miou_no_bg"])
    history["val_mf1"].append(val_scores["mf1"])
    history["val_disaster_acc"].append(val_disaster_scores["accuracy"])
    history["val_iou_per_class"].append(val_scores["iou_per_class"])
    history["val_f1_per_class"].append(val_scores["f1_per_class"])
    history["lr"].append(optimizer.param_groups[0]["lr"])
    history["epoch_time"].append(epoch_time)

    print(
        f"Epoch {epoch:3d}/{NUM_EPOCHS} | "
        f"tr_loss={train_losses['total']:.3f} (F={train_losses['focal']:.2f} D={train_losses['dice']:.2f} B={train_losses['boundary']:.2f}) | "
        f"val_loss={val_loss_avg:.3f} | "
        f"val_mIoU={val_scores['miou']:.3f} | "
        f"val_mIoU_no_bg={val_scores['miou_no_bg']:.3f} | "
        f"LR={optimizer.param_groups[0]['lr']:.1e} | "
        f"{epoch_time:.1f} dk"
    )

    # Per-class IoU (her 5 epoch)
    if epoch % 5 == 0 or epoch == 1:
        iou_per = val_scores["iou_per_class"]
        print(f"  Per-class IoU:", end=" ")
        for i, (name, v) in enumerate(zip(class_names, iou_per)):
            print(f"{name[:4]}={v:.2f}", end=" ")
        print()

    # Checkpoint
    is_best = val_scores["miou_no_bg"] > best_miou_no_bg
    if is_best:
        best_miou_no_bg = val_scores["miou_no_bg"]
        best_path = os.path.join(CKPT_TEACHER, "teacher_v2_best.pth")
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "val_miou": val_scores["miou"],
            "val_miou_no_bg": val_scores["miou_no_bg"],
            "history": history,
        }, best_path)
        print(f"  🏆 Yeni en iyi! mIoU_no_bg={best_miou_no_bg:.4f} kaydedildi")

    # Her 10 epoch backup
    if epoch % 10 == 0:
        backup_path = os.path.join(CKPT_TEACHER, f"teacher_v2_epoch_{epoch}.pth")
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "history": history,
        }, backup_path)
        print(f"  💾 Backup: teacher_v2_epoch_{epoch}.pth")

    # History JSON (her epoch)
    history_path = os.path.join(CKPT_TEACHER, "training_history_v2.json")
    with open(history_path, "w") as f:
        json.dump(history, f, indent=2)

print("\n" + "=" * 75)
print(f"🎉 EĞİTİM TAMAMLANDI (v2)")
print(f"   En iyi val_mIoU_no_bg: {best_miou_no_bg:.4f}")
print(f"   Hedef:                  0.78")
if best_miou_no_bg >= 0.78:
    print(f"   ✅ HEDEF BAŞARILDI!")
elif best_miou_no_bg >= 0.55:
    print(f"   📊 İyi başlangıç, Phase 2 (fine-tune) ile daha da iyileştirilebilir")
else:
    print(f"   ⚠️  Beklenenden düşük, tanı gerekli")
print(f"   En iyi model: {os.path.join(CKPT_TEACHER, 'teacher_v2_best.pth')}")


🚀 EĞİTİM BAŞLIYOR (v2)
   Toplam epoch: 80
   Train batch:  122
   Val batch:    27
   Checkpoint:   /content/drive/MyDrive/AFETSONAR/checkpoints/teacher


Epoch 1/80 [train]:   0%|          | 0/122 [01:19<?, ?it/s]

Epoch 1/80 [val]  :   0%|          | 0/27 [00:00<?, ?it/s]

Epoch   1/80 | tr_loss=0.925 (F=0.47 D=0.98 B=1.73) | val_loss=0.777 | val_mIoU=0.134 | val_mIoU_no_bg=0.037 | LR=3.4e-05 | 10.6 dk
  Per-class IoU: bg=0.62 no_d=0.07 mino=0.02 majo=0.03 dest=0.07 un-c=0.00 
  🏆 Yeni en iyi! mIoU_no_bg=0.0366 kaydedildi


Epoch 2/80 [train]:   0%|          | 0/122 [00:00<?, ?it/s]

...

Epoch  78/80 | tr_loss=0.117 (F=0.07 D=0.32 B=0.11) | val_loss=0.239 | val_mIoU=0.432 | val_mIoU_no_bg=0.324 | LR=1.0e-10 | 0.8 dk


Epoch 79/80 [train]:   0%|          | 0/122 [00:00<?, ?it/s]

Epoch 79/80 [val]  :   0%|          | 0/27 [00:00<?, ?it/s]

Epoch  79/80 | tr_loss=0.121 (F=0.07 D=0.34 B=0.11) | val_loss=0.241 | val_mIoU=0.430 | val_mIoU_no_bg=0.322 | LR=4.0e-08 | 0.8 dk


Epoch 80/80 [train]:   0%|          | 0/122 [00:00<?, ?it/s]

## 7️⃣ Eğitim Eğrileri

In [ ]:
import matplotlib.pyplot as plt
import json

history_path = os.path.join(CKPT_TEACHER, "training_history_v2.json")
with open(history_path) as f:
    history = json.load(f)

epochs = list(range(1, len(history["train_loss"]) + 1))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Train/Val loss
ax = axes[0, 0]
ax.plot(epochs, history["train_loss"], label="train", color="C0")
ax.plot(epochs, history["val_loss"], label="val", color="C1")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.set_title("Train vs Val Loss")
ax.legend(); ax.grid(alpha=0.3)

# mIoU
ax = axes[0, 1]
ax.plot(epochs, history["val_miou"], label="val mIoU", color="C2")
ax.plot(epochs, history["val_miou_no_bg"], label="val mIoU (no bg)", color="C3", linewidth=2)
ax.axhline(y=0.78, color="red", linestyle="--", alpha=0.5, label="hedef 0.78")
ax.axhline(y=0.55, color="orange", linestyle="--", alpha=0.5, label="eşik 0.55")
ax.set_xlabel("Epoch"); ax.set_ylabel("mIoU")
ax.set_title("Validation mIoU")
ax.legend(); ax.grid(alpha=0.3)

# Loss components
ax = axes[0, 2]
ax.plot(epochs, history["train_focal"], label="Focal", color="C0")
ax.plot(epochs, history["train_dice"], label="Dice", color="C1")
ax.plot(epochs, history["train_boundary"], label="Boundary", color="C2")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.set_title("Damage Loss Komponentleri")
ax.legend(); ax.grid(alpha=0.3)

# Multi-task losses
ax = axes[1, 0]
ax.plot(epochs, history["train_change"], label="Change (CE)", color="C4")
ax.plot(epochs, history["train_disaster"], label="Disaster (CE)", color="C5")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.set_title("Auxiliary Task Losses")
ax.legend(); ax.grid(alpha=0.3)

# Per-class IoU (son 10 epoch ortalaması, bar chart)
ax = axes[1, 1]
n_last = min(10, len(history["val_iou_per_class"]))
last_iou = [history["val_iou_per_class"][-(i+1)] for i in range(n_last)]
import numpy as np
mean_iou = np.nanmean(last_iou, axis=0)
colors = ["black", "green", "yellow", "orange", "red", "purple"]
ax.bar(range(6), mean_iou, color=colors)
ax.set_xticks(range(6))
ax.set_xticklabels(["bg", "no", "minor", "major", "dest", "uncls"])
ax.set_ylabel("IoU")
ax.set_title(f"Per-Class IoU (son {n_last} epoch ortalaması)")
ax.grid(alpha=0.3, axis="y")
for i, v in enumerate(mean_iou):
    if not np.isnan(v):
        ax.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=9)

# Learning rate
ax = axes[1, 2]
ax.plot(epochs, history["lr"], color="C6")
ax.set_xlabel("Epoch"); ax.set_ylabel("LR")
ax.set_title("LR Schedule (warmup + cosine)")
ax.set_yscale("log")
ax.grid(alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(OUTPUTS_VIZ, "teacher_v2_training_curves.png")
plt.savefig(fig_path, dpi=100, bbox_inches="tight")
print(f"✅ Kaydedildi: {fig_path}")
plt.show()

# Özet
print(f"\n📊 Özet:")
print(f"   En iyi val_mIoU_no_bg: {max(history['val_miou_no_bg']):.4f}")
print(f"   En iyi val_mIoU:       {max(history['val_miou']):.4f}")
print(f"   Son epoch mIoU_no_bg:  {history['val_miou_no_bg'][-1]:.4f}")
print(f"   Toplam eğitim süresi:  {sum(history['epoch_time']):.1f} dakika")

## 8️⃣ Örnek Tahminler

Val setinden 4 örnek üzerinde tahmin yap. Özellikle hasarlı örnekleri seç.

In [ ]:
import sys
sys.path.insert(0, SRC_DIR)
import importlib
import utils
importlib.reload(utils)
from utils import DAMAGE_COLORS, denormalize
from matplotlib.colors import ListedColormap

# v2 için 6 renk
CMAP_V2 = ListedColormap(np.array([
    [0, 0, 0], [0, 255, 0], [255, 255, 0], [255, 128, 0], [255, 0, 0], [200, 0, 200]
]) / 255.0)

# En iyi modeli yükle
best_path = os.path.join(CKPT_TEACHER, "teacher_v2_best.pth")
ckpt = torch.load(best_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"✅ Best v2 yüklendi (epoch {ckpt['epoch']}, mIoU_no_bg={ckpt['val_miou_no_bg']:.4f})")

# 4 hasarlı örnek bul
import pandas as pd
val_df = pd.read_csv(os.path.join(DATA_SPLITS, "val_v2.csv"))
damaged_df = val_df[val_df.get("has_any_damage", val_df.get("damage_present", False))]

# Varsa damage_present column, yoksa has_any_damage
if len(damaged_df) == 0:
    damaged_df = val_df  # fallback

import random
random.seed(42)
sample_indices = random.sample(range(len(val_ds)), min(4, len(val_ds)))

fig, axes = plt.subplots(4, 4, figsize=(20, 20))
amp_dtype = torch.bfloat16 if PRECISION == "bf16" else torch.float16

for row, idx in enumerate(sample_indices):
    sample = val_ds[idx]
    img_batch = sample["image"].unsqueeze(0).to(device)

    with torch.no_grad():
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            out = model(img_batch)
        pred = out["damage_logits"].argmax(dim=1)[0].cpu()

    pre = denormalize(sample["image"][:3]).permute(1, 2, 0).numpy().clip(0, 1)
    post = denormalize(sample["image"][3:]).permute(1, 2, 0).numpy().clip(0, 1)

    axes[row, 0].imshow(pre); axes[row, 0].set_title("Pre" if row==0 else ""); axes[row, 0].axis("off")
    axes[row, 1].imshow(post); axes[row, 1].set_title("Post" if row==0 else ""); axes[row, 1].axis("off")
    axes[row, 2].imshow(sample["mask"].numpy(), cmap=CMAP_V2, vmin=0, vmax=5)
    axes[row, 2].set_title("GT" if row==0 else ""); axes[row, 2].axis("off")
    axes[row, 3].imshow(pred.numpy(), cmap=CMAP_V2, vmin=0, vmax=5)
    axes[row, 3].set_title("Prediction" if row==0 else ""); axes[row, 3].axis("off")

plt.suptitle("Teacher v2 — Validation Predictions", fontsize=14, y=1.001)
plt.tight_layout()
pred_path = os.path.join(OUTPUTS_VIZ, "teacher_v2_predictions.png")
plt.savefig(pred_path, dpi=80, bbox_inches="tight")
print(f"✅ Kaydedildi: {pred_path}")
plt.show()

## 🎉 Notebook 2_v2 Tamamlandı!

### Ne yaptık?

✅ 8 iyileştirmenin tümü uygulandı
✅ Siamese SegFormer-B3 inşa edildi (~45M param)
✅ Combo Loss (Focal + Dice + Boundary)
✅ Agresif class weights + WeightedRandomSampler
✅ Building-aware crop
✅ Sınıf-koruyucu augmentation
✅ 80 epoch eğitim

### Drive'da

```
AFETSONAR/
├── checkpoints/teacher/
│   ├── teacher_best.pth              ← ESKİ (0.298 mIoU)
│   ├── teacher_v2_best.pth           ← YENİ
│   ├── teacher_v2_epoch_*.pth        ← Yedekler
│   └── training_history_v2.json
├── outputs/visualizations/
│   ├── teacher_v2_training_curves.png
│   └── teacher_v2_predictions.png
└── src/
    ├── models_v2.py
    ├── losses_v2.py
    ├── dataset_v2.py
    └── augmentations_v2.py
```

### Sonraki Adımlar

**Eğer mIoU_no_bg ≥ 0.78:**
🎉 Hedefe ulaşıldı! Öğrenci modeli (Notebook 3) ile distillation'a geçebiliriz.

**Eğer 0.55 ≤ mIoU_no_bg < 0.78:**
📊 İyi ilerleme. Şu opsiyonlar var:
- Notebook 2_v3: Phase 2 fine-tuning (daha fazla epoch, daha düşük LR)
- Notebook 3'e geç ama daha dikkatli, distillation'ın öğretmenin sınırlarını aşamayacağını unutmadan

**Eğer mIoU_no_bg < 0.55:**
⚠️ Sorunlar devam ediyor. 02b tanı testini tekrar çalıştır, yeni tanı gerekli.

### ⚠️ Bana ne göndereceksin

Eğitim bittiğinde:
1. Son 10 epoch'un çıktısı (console loglar)
2. `teacher_v2_training_curves.png`
3. `teacher_v2_predictions.png`
4. Son epoch'un **per-class IoU** satırı

Bunları gördüğümde next step'e karar veririz.

---

**Calamitas AI · Teknofest 2025 · Notebook 2_v2** 🚀